## Importing Necessary Libraries

In [978]:
import re
from distutils.fancy_getopt import wrap_text

from ipykernel.pickleutil import istype
from mpmath import isint
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer, PorterStemmer
import emoji
import nltk
from spellchecker import SpellChecker
from googletrans import Translator
import pandas as pd
from autocorrect import Speller
from deep_translator import GoogleTranslator
import tqdm
from tqdm.auto import tqdm
import datetime
from langdetect import detect
import spacy
from datetime import datetime, timedelta
import sqlite3
import isodate

In [809]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/sagnikchakravarty/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/sagnikchakravarty/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/sagnikchakravarty/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/sagnikchakravarty/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/sagnikchakravarty/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

# Creating Functions for Data Cleaning

The data cleaning class would clean each column for a dataframe:

- It ignores any proper noun
- it removes stop words but it keeps emojis
- it translates language to english using google translator
- it corrects any spelling spelling errors
- then finally it does stemming and lemmatization

This is the ultimate text cleaner i would say

For the Sentiment analysis we are using both removing stopwords, stemming and lemmatization while for Topic Modeling we are only removing stopwords and basic text preprocessing


In [645]:
class TextCleaner:
    def __init__(self, remove_stopwords=True, stemming=True, lemmatize=True, autocorrect = True, translation = True):
        """
        Initialize the TextCleaner class with optional configurations.
        :param remove_stopwords: Whether to remove stopwords (default: True)
        :param stemming: Whether to apply stemming (default: False)
        :param lemmatize: Whether to apply lemmatization (default: True)
        :param target_language: Target language to keep (default: english)
        """
        self.translator = GoogleTranslator(source='auto', target='en')
        self.remove_stopwords = remove_stopwords
        self.stemming = stemming
        self.stemmer = PorterStemmer()
        self.spell = Speller(lang='en')
        self.nlp = spacy.load("en_core_web_sm")  # Load spaCy model
        self.remove_stopwords = remove_stopwords
        self.lemmatize = lemmatize
        self.autocorrect = autocorrect
        self.translation = translation
        
    def translate_large_text(self, text):
        """
        Translate large text by splitting it into chunks of 5000 characters.
        If the detected language is English, skip translation.
        :param text: The input text to be translated.
        :return: Translated text as a single string.
        """
        try:
            # Detect the language of the input text
            detected_language = detect(text)
            
            # If the detected language is English, return the original text
            if detected_language == 'en':
                return text
    
            # Split the text into chunks of 5000 characters
            max_chunk_size = 5000
            chunks = [text[i:i + max_chunk_size] for i in range(0, len(text), max_chunk_size)]
    
            # Translate each chunk and combine the results
            translated_chunks = [self.translator.translate(chunk) for chunk in chunks]
            return " ".join(translated_chunks)
    
        except Exception as e:
            print(f"Translation failed: {e}")
            return text

        
    def clean_text(self, text):
        """
        Clean a single text string by applying various preprocessing steps.
        :param text: The input text string.
        :return: The cleaned text string.
        """
        # Handle empty strings early
        if not text.strip():
            return ''  # Leave empty strings blank
        # To lower case
        text = text.lower()
        
        # Translation
        if self.translation:
            text = self.translate_large_text(text)
        # Remove URLs
        text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)

        # Remove HTML tags
        text = re.sub(r'<.*?>', '', text)
        
        # Convert emojis to Unicode representation using the emoji library
        text = emoji.demojize(text)
        
        text = re.sub(r'[^a-zA-Z\s]', '', text)

        doc = self.nlp(text)
        
        words = [
            token.text for token in doc 
            if token.pos_ == "PROPN" or (not token.is_stop and token.is_alpha)
        ]
        
        # spell checker
        if self.autocorrect:
            words = [self.spell(word) for word in words]


        # Apply stemming if enabled
        if self.stemming:
            words = [self.stemmer.stem(word) for word in words]

        # Apply lemmatization if enabled
        if self.lemmatize:
            words = [token.lemma_ for token in doc if token.text in words]

        return ' '.join(words)

    def clean_dataframe(self, df, columns):
        """
        Clean specified columns in a DataFrame.
        :param df: The input DataFrame.
        :param columns: List of column names to clean.
        :return: The cleaned DataFrame.
        """
        
        try:
            if type(columns) == str:
                columns = [columns]
            for column in columns:
                tqdm.pandas(desc = f'cleaning data {column}', total = len(df[column]), colour = 'blue')
                if column not in df.columns:
                    raise ValueError(f"Column '{column}' does not exist in the DataFrame.")

                if not pd.api.types.is_string_dtype(df[column]):
                    raise TypeError(f"Column '{column}' is not a text column. Found type: {df[column].dtype}")
                
                df[column] = df[column].progress_apply(self.clean_text)
            print("Successfully cleaned data frame.")

            return df

        except Exception as e:
            print(f"Error: {e}")
            return df

In [810]:
sentiment_cleaner = TextCleaner(remove_stopwords=True, stemming=True, lemmatize= True, translation=False, autocorrect=False)
topicmodel_cleaner = TextCleaner(remove_stopwords=True, stemming=False, lemmatize=False, autocorrect = False, translation = False)

In [893]:
keywords = ['young thug', 
            'gunna', 
            'yfn lucci',
            'yfn',
            'ysl', 
            'ysl rico',
            'rico',
            'yak gotti',
            'yak',
            'gotti',
            'shannon stillwell',
            'shannon',
            'stillwell',
            'sb',
            'jeffrey williams',
            'jeffrey',
            'williams',
            'deamonte kendrick',
            'deamonte',
            'kendrick',
            'sergio kitchens',
            'sergio',
            'kitchens',
            'lil rod',
            'rodalius ryan',
            'rodalius',
            'ryan'
            'qua',
            'quamarvious nichols',
            'quamarvious',
            'nichols',
            'christian eppinger',
            'eppinger',
            'bhris',
            'paige reese whitaker',
            'whitaker',
            'ural d. glanville',
            'ural glanville',
            'glanville',
            'shukura l. ingram',
            'ingram',
            'brian steel',
            'adriane love',
            'simone hylton',
            'simone',
            'hylton',
            'thuggerdaily'
            ]
def find_matching_keywords(title, keywords, replace = None):
    matches = ', '.join([keyword for keyword in keywords if keyword.lower() in title.lower()])
    return matches if matches else replace

# 1. Newspaper

The newspaper from which we have collected the data are as follows:

- New York Times
- Atlanta Journal Conference
- New York Post
- Washington Post

## 1.1 New York Times

In [897]:
# YOUNG THUG
young_thug_nyt = pd.read_excel('/Users/sagnikchakravarty/Documents/Young Thug/Data/UnCleaned Data/News paper/nyt_article_result.xlsx',
                               'Young Thug')
young_thug_nyt['Search Term'] = 'Young Thug'

#GUNNA
gunna_nyt = pd.read_excel('/Users/sagnikchakravarty/Documents/Young Thug/Data/UnCleaned Data/News paper/nyt_article_result.xlsx',
                               'Gunna')
gunna_nyt['Search Term'] = 'Gunna'

# YFN LUCCI
yfn_nyt = pd.read_excel('/Users/sagnikchakravarty/Documents/Young Thug/Data/UnCleaned Data/News paper/nyt_article_result.xlsx',
                               'YFN Lucci')
yfn_nyt['Search Term'] = 'YFN Lucci'

# YSL
ysl_nyt = pd.read_excel('/Users/sagnikchakravarty/Documents/Young Thug/Data/UnCleaned Data/News paper/nyt_article_result.xlsx',
                               'YSL')
ysl_nyt['Search Term'] = 'YSL'

# YSL RICO
yslrico_nyt = pd.read_excel('/Users/sagnikchakravarty/Documents/Young Thug/Data/UnCleaned Data/News paper/nyt_article_result.xlsx',
                               'YSL RICO')

yslrico_nyt['Search Term'] = 'YSL RICO'

nyt = pd.concat([young_thug_nyt, gunna_nyt, ysl_nyt, yfn_nyt, yslrico_nyt])
nyt['Source'] = 'New York Times'
nyt['publication_date'] = pd.to_datetime(nyt['publication_date']).dt.date


nyt['article'] = nyt['lead_paragraph']
nyt['headline'] = nyt['abstract']
nyt = nyt.drop(['abstract', 'lead_paragraph', 'word_count'], axis = 1)
nyt = nyt.groupby('article', as_index=False).agg({
    'Search Term': ', '.join,  # Merge 'Search Term' values
    'publication_date': 'first',  # Keep the first publication date
    'snippet': 'first',  # Keep the first snippet
    'Source': 'first',  # Keep the first source
    'headline': 'first'
})
nyt = nyt[['publication_date', 'headline','snippet', 'article','Search Term', 'Source']]

In [898]:
nyt

,publication_date,headline,snippet,article,Search Term,Source
0,2024-10-29,Prosecutors have accused the star Atlanta rapp...,Prosecutors have accused the star Atlanta rapp...,A defendant in the racketeering and gang consp...,"YSL, YSL RICO",New York Times
1,2023-11-09,A judge ruled on Thursday that at least 17 spe...,A judge ruled on Thursday that at least 17 spe...,A judge decided on Thursday that rap lyrics by...,YSL RICO,New York Times
2,2024-06-11,"Brian Steel, a lawyer for the Atlanta rapper, ...","Brian Steel, a lawyer for the Atlanta rapper, ...",A star witness jailed for refusing to testify....,YSL RICO,New York Times
3,2022-08-11,As top allies of Donald J. Trump are called to...,As top allies of Donald J. Trump are called to...,ATLANTA — Amid a deepening swirl of federal an...,YFN Lucci,New York Times
4,2023-01-13,"With a blockbuster case set to begin, the supe...","With a blockbuster case set to begin, the supe...","ATLANTA — Day after day, the young men came be...",YFN Lucci,New York Times
5,2022-05-10,"He was indicted with about two dozen others, i...","He was indicted with about two dozen others, i...","ATLANTA — The rap star Young Thug, one of the ...",YFN Lucci,New York Times
6,2024-07-01,The much-delayed case was halted indefinitely ...,The much-delayed case was halted indefinitely ...,After more than 10 months of jury selection an...,YSL RICO,New York Times
7,2024-11-08,Did you follow the news this week? Take our qu...,Did you follow the news this week? Take our qu...,Did you follow the news this week? Take our qu...,Young Thug,New York Times
8,2024-01-30,"The stones in the teeth of Post Malone, Odell ...","The stones in the teeth of Post Malone, Odell ...",Dr. Thomas Connelly has turned the cliché “mil...,Gunna,New York Times
9,2024-11-12,How far the state’s election interference case...,How far the state’s election interference case...,Even as the federal criminal cases against Don...,Young Thug,New York Times


## 1.2 Washington Post

In [899]:
# YOUNG THUG
young_thug_wp = pd.read_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/UnCleaned Data/News paper/Washington Post/youngthug_wp.csv')
young_thug_wp['Source'] = 'Washington Post'
young_thug_wp['Search Term'] = 'Young Thug'
young_thug_wp.rename(columns={'date': 'publication_date'}, inplace=True)
young_thug_wp.drop('links', axis=1, inplace=True)

# Gunna
gunna_wp = pd.read_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/UnCleaned Data/News paper/Washington Post/gunna_wp.csv')
gunna_wp['Source'] = 'Washington Post'
gunna_wp['Search Term'] = 'Gunna'
gunna_wp.rename(columns={'date': 'publication_date'}, inplace=True)
gunna_wp.drop('links', axis=1, inplace=True)

# YSL
ysl_wp = pd.read_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/UnCleaned Data/News paper/Washington Post/ysl_wp.csv')
ysl_wp['Source'] = 'Washington Post'
ysl_wp['Search Term'] = 'YSL'
ysl_wp.rename(columns={'date': 'publication_date'}, inplace=True)
ysl_wp.drop('links', axis=1, inplace=True)

# YSL RICO
yslrico_wp = pd.read_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/UnCleaned Data/News paper/Washington Post/yslrico_wp.csv')
yslrico_wp['Source'] = 'Washington Post'
yslrico_wp['Search Term'] = 'YSL Rico'
yslrico_wp.rename(columns={'date': 'publication_date'}, inplace=True)
yslrico_wp.drop('links', axis=1, inplace=True)

# YFN RICO
yfn_wp = pd.read_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/UnCleaned Data/News paper/Washington Post/yfnlucci_wp.csv')
yfn_wp['Source'] = 'Washington Post'
yfn_wp['Search Term'] = 'YFN Lucci'
yfn_wp.rename(columns={'date': 'publication_date'}, inplace=True)
yfn_wp.drop('links', axis=1, inplace=True)

In [900]:
wp = pd.concat([young_thug_wp, gunna_wp, ysl_wp, yslrico_wp, yfn_wp])
wp = wp.groupby('article', as_index=False).agg({
    'Search Term': ', '.join,  # Merge 'Search Term' values
    'publication_date': 'first',  # Keep the first publication date
    'snippet': 'first',  # Keep the first snippet
    'Source': 'first',  # Keep the first source
    'headline': 'first'
})
wp = wp[['publication_date', 'headline','snippet', 'article','Search Term', 'Source']]
wp['publication_date'] = pd.to_datetime(wp['publication_date']).dt.date
wp['article'] = wp['article'].astype('str')
wp['headline'] = wp['headline'].astype('str')
wp['snippet'] = wp['snippet'].astype('str')

In [901]:
wp

,publication_date,headline,snippet,article,Search Term,Source
0,2018-09-14,‘Latinx’: An offense to the Spanish language o...,The gender-neutral word has become so popular ...,\nAbout US is a new initiative by The Washingt...,YSL Rico,Washington Post
1,2024-01-09,Joseph Ferguson joins The Post’s Video team as...,Joseph Ferguson joins Video as a TikTok host a...,\nAnnouncement from Director of Video Micah Ge...,YSL Rico,Washington Post
2,2019-07-17,Of course Lil Nas X has a new ‘Old Town Road’ ...,It doesn't matter how many times you've heard ...,\nLil Nas X has released his third “Old Town R...,Young Thug,Washington Post
3,2019-02-10,Grammy Awards 2019 FAQ: How to live-stream the...,"Everything you need to know, from the performe...",(All times Eastern)\nWhere to watch on TV:\n,Young Thug,Washington Post
4,2017-09-24,TV highlights: ‘Young Sheldon’ premieres on CBS,"Monday, Sept. 25, 2017 | Also: ‘The Good Docto...",(All times Eastern.)\nDancing With the Stars ...,Young Thug,Washington Post
...,...,...,...,...,...,...
276,2018-08-20,MTV VMAs: Where to watch the show and red carp...,"Jennifer Lopez, Nicki Minaj, Ariana Grande, Ca...",Where can I watch the VMAs?\nThe 2018 MTV Vide...,Young Thug,Washington Post
277,2018-04-11,4 concerts to catch in the Washington area ove...,"See YFN Lucci, Lucy Dacus, U.S. Girls and Pink...",YFN Lucci\nFresh off the release of his debut ...,YFN Lucci,Washington Post
278,2024-03-27,PerspectiveIs Young Thug on trial for using hi...,"In America, sadly, being Black means your imag...",Young Thug’s 2019 album “So Much Fun” opens wi...,"Young Thug, Gunna, YSL, YSL Rico",Washington Post
279,2023-11-27,Young Thug’s rap lyrics take center stage as t...,Rapper Young Thug’s racketeering trial began t...,correctionA previous version of this story and...,"Young Thug, Gunna, YSL Rico",Washington Post


## 1.3 AJC

In [902]:
# YOUNG THUG
youngthug_ajc = pd.read_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/UnCleaned Data/News paper/AJC/ajc_youngthug.csv')
youngthug_ajc['Search Term'] = 'Young Thug'

# GUNNA
gunna_ajc = pd.read_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/UnCleaned Data/News paper/AJC/ajc_gunna.csv')
gunna_ajc['Search Term'] = 'Gunna'

# YSL
ysl_ajc = pd.read_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/UnCleaned Data/News paper/AJC/ajc_ysl.csv')
ysl_ajc['Search Term'] = 'YSL'

# YSL RICO
yslrico_wp = pd.read_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/UnCleaned Data/News paper/AJC/ajc_yslrico.csv')
yslrico_wp['Search Term'] = 'YSL Rico'

# YFN LUCCI
yfn_ajc = pd.read_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/UnCleaned Data/News paper/AJC/ajc_yfnlucci.csv')
yfn_ajc['Search Term'] = 'YFN Lucci'

ajc = pd.concat([youngthug_ajc, gunna_ajc, ysl_ajc, yfn_ajc, yfn_ajc])
ajc

,headline,publication_date,snippet,links,article,Search Term
0,Atlanta area remains off-limits to Young Thug ...,"Dec 11, 2024",A Fulton County judge has denied Young Thug’s ...,/news/crime/atlanta-area-remains-off-limits-to...,A Fulton County judge has denied Young Thug’s ...,Young Thug
1,Home for the holidays? Young Thug asks judge t...,"Dec 11, 2024",Young Thug is asking a Fulton County judge to ...,/news/crime/home-for-the-holidays-young-thug-a...,Young Thug is asking a Fulton County judge to ...,Young Thug
2,Young Thug trial: Here’s what went wrong for F...,"Dec 06, 2024","In the spring of 2022, Fulton County District ...",/news/crime/young-thug-trial-heres-what-went-w...,"In the spring of 2022, Fulton County District ...",Young Thug
3,Young Thug trial: Case headed to jury today af...,"Nov 26, 2024",Nearly two years after jury selection began in...,/news/crime/jury-expected-to-begin-deliberatio...,Nearly two years after jury selection began in...,Young Thug
4,Young Thug: Prosecutors rest their case in Geo...,"Nov 19, 2024",After nearly a year’s worth of testimony from ...,/news/crime/prosecutors-rest-their-case-agains...,After nearly a year’s worth of testimony from ...,Young Thug
...,...,...,...,...,...,...
50,High school girls track player of the year,"May 31, 2017","Girls track player of the year Kennedy Simon, ...",/sports/high-school/high-school-girls-track-pl...,Girls track player of the year\n\nKennedy Simo...,YFN Lucci
51,Streetz 94.5's newest night host: Ferrari Simm...,"Jan 10, 2017","This is posted on Tuesday, January 10, 2017 by...",/blog/radiotvtalk/streetz-newest-night-host-fe...,"This is posted on Tuesday, January 10, 2017 by...",YFN Lucci
52,"Radio briefs: Erick Erickson, August/September...","Sep 28, 2016","By RODNEY HO/ rho@ajc.com, originally filed Tu...",/blog/radiotvtalk/radio-briefs-erick-erickson-...,"By RODNEY HO/ rho@ajc.com, originally filed Tu...",YFN Lucci
53,"Radio briefs: Hot 107.9 on 'The Rap Game,' Aug...","Sep 04, 2016","By RODNEY HO/ rho@ajc.com, originally filed Su...",/blog/radiotvtalk/radio-briefs-hot-107-the-rap...,"By RODNEY HO/ rho@ajc.com, originally filed Su...",YFN Lucci


In [903]:
ajc['publication_date'] = pd.to_datetime(ajc['publication_date']).dt.date
ajc['article'] = ajc['article'].astype('str')
ajc['headline'] = ajc['headline'].astype('str')
ajc['snippet'] = ajc['snippet'].astype('str')
ajc['Source'] = 'AJC'
ajc.drop('links', axis=1, inplace=True)
ajc = ajc.groupby('article', as_index=False).agg({
    'Search Term': ', '.join,  # Merge 'Search Term' values
    'publication_date': 'first',  # Keep the first publication date
    'snippet': 'first',  # Keep the first snippet
    'Source': 'first',  # Keep the first source
    'headline': 'first'
})
ajc = ajc[['publication_date', 'headline','snippet', 'article','Search Term', 'Source']]

## 1.4 New York Post

In [904]:
# Young Thug
youngthug_nyp = pd.read_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/UnCleaned Data/News paper/New York Post/young_thug_nyp.csv')
youngthug_nyp['Search Term'] = 'Young Thug'

# GUNNA
gunna_nyp = pd.read_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/UnCleaned Data/News paper/New York Post/gunna_nyp.csv')
gunna_nyp['Source'] = 'Gunna'

# YSL
ysl_nyp = pd.read_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/UnCleaned Data/News paper/New York Post/ysl_nyp.csv')
ysl_nyp['Search Term'] = 'YSL'

# YSL RICO
yslrico_nyp = pd.read_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/UnCleaned Data/News paper/New York Post/ysl_rico_nyp.csv')
yslrico_nyp['Search Term'] = 'YSL Rico'

# YFN LUCCI
yfn_nyp = pd.read_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/UnCleaned Data/News paper/New York Post/yfn_lucci_nyp.csv')
yfn_nyp['Search Term'] = 'YFN Lucci'

nyp = pd.concat([youngthug_nyp, ysl_nyp, yfn_nyp, yslrico_nyp, gunna_nyp])

In [905]:
nyp['Source'] = 'New York Post'
nyp['headline'] = nyp['headline'].astype('str')
nyp['snippet'] = nyp['snippet'].astype('str')
nyp['article'] = nyp['article'].astype('str')
nyp.rename(columns={'date': 'publication_date'}, inplace=True)
nyp['publication_date'] = pd.to_datetime(nyp['publication_date']).dt.date
nyp.drop('links', axis=1, inplace=True)

In [906]:
nyp = nyp.groupby('article', as_index=False).agg({
    'Search Term': lambda x: ', '.join(filter(None, map(str, x))),  # Merge 'Search Term' values
    'publication_date': 'first',  # Keep the first publication date
    'snippet': 'first',  # Keep the first snippet
    'Source': 'first',  # Keep the first source
    'headline': 'first'
})
nyp = nyp[['publication_date', 'headline','snippet', 'article','Search Term', 'Source']]

In [907]:
newspaper = pd.concat([wp, ajc, nyp])

In [908]:
newspaper['complete'] = newspaper['headline']+' '+ newspaper['snippet']+' '+ newspaper['article']
newspaper['Keywords'] = newspaper['complete'].apply(lambda article: find_matching_keywords(article, keywords))
newspaper.drop('complete', axis=1, inplace=True)
newspaper

,publication_date,headline,snippet,article,Search Term,Source,Keywords
0,2018-09-14,‘Latinx’: An offense to the Spanish language o...,The gender-neutral word has become so popular ...,\nAbout US is a new initiative by The Washingt...,YSL Rico,Washington Post,None
1,2024-01-09,Joseph Ferguson joins The Post’s Video team as...,Joseph Ferguson joins Video as a TikTok host a...,\nAnnouncement from Director of Video Micah Ge...,YSL Rico,Washington Post,None
2,2019-07-17,Of course Lil Nas X has a new ‘Old Town Road’ ...,It doesn't matter how many times you've heard ...,\nLil Nas X has released his third “Old Town R...,Young Thug,Washington Post,None
3,2019-02-10,Grammy Awards 2019 FAQ: How to live-stream the...,"Everything you need to know, from the performe...",(All times Eastern)\nWhere to watch on TV:\n,Young Thug,Washington Post,None
4,2017-09-24,TV highlights: ‘Young Sheldon’ premieres on CBS,"Monday, Sept. 25, 2017 | Also: ‘The Good Docto...",(All times Eastern.)\nDancing With the Stars ...,Young Thug,Washington Post,young thug
...,...,...,...,...,...,...,...
1026,None,Christina Hall ditches wedding ring as she enj...,The HGTV star completed her look with sunglass...,“Christina at the Spa” has a nice ring to it.\...,YSL,New York Post,"ysl, sb"
1027,None,NYC landlord case shows DA Alvin Bragg's unequ...,Bragg’s hyper-selective prosecutions prove tha...,"“Everyone stands equal before the law,” Manhat...",Young Thug,New York Post,None
1028,None,"Madonna, 64, rocks lingerie and a riding crop ...","The ""Material Girl"" hitmaker celebrated the re...","“Express Yourself,” indeed.\n\nMadonna was the...",YSL,New York Post,ysl
1029,None,Failure to launch?,“I had been presented — for no right reason — ...,“I had been presented — for no right reason — ...,YSL,New York Post,ysl


In [909]:
newspaper.dropna(subset=['Keywords'], inplace=True)
newspaper

,publication_date,headline,snippet,article,Search Term,Source,Keywords
4,2017-09-24,TV highlights: ‘Young Sheldon’ premieres on CBS,"Monday, Sept. 25, 2017 | Also: ‘The Good Docto...",(All times Eastern.)\nDancing With the Stars ...,Young Thug,Washington Post,young thug
7,2022-01-14,What to watch on Friday: ‘The House’ on Netflix,Chillin Island (HBO at 10:30) Gunna tries to s...,(All times Eastern.)\nShark Tank (ABC at 8) Th...,Gunna,Washington Post,gunna
10,2017-04-25,TV highlights: ‘The Handmaid’s Tale’ comes to ...,"Wednesday, April 26, 2017| Rashida Jones guest...",(All times Eastern.)\nSurvivor: Game Changers ...,Young Thug,Washington Post,young thug
13,2015-10-01,Easy as 1-B-iii: 4 formats x 4 subjects x 4 li...,Winning parodies abound in our 64-option conte...,(Click here to skip down to this week's new c...,YSL,Washington Post,sb
14,2022-05-10,Tuesday briefing: Russia’s quiet Victory Day; ...,Catch up in minutes with these 7 stories.,1\nRussia’s Victory Day was much quieter than ...,Young Thug,Washington Post,young thug
...,...,...,...,...,...,...,...
1023,None,'30 Rock' gives us Moore to love,"""30 Rock"" is giving ""The Love Boat"" a run for ...",“30 Rock” is giving “The Love Boat” a run for ...,nan,New York Post,gunna
1026,None,Christina Hall ditches wedding ring as she enj...,The HGTV star completed her look with sunglass...,“Christina at the Spa” has a nice ring to it.\...,YSL,New York Post,"ysl, sb"
1028,None,"Madonna, 64, rocks lingerie and a riding crop ...","The ""Material Girl"" hitmaker celebrated the re...","“Express Yourself,” indeed.\n\nMadonna was the...",YSL,New York Post,ysl
1029,None,Failure to launch?,“I had been presented — for no right reason — ...,“I had been presented — for no right reason — ...,YSL,New York Post,ysl


In [910]:
newspaper_sentiment = sentiment_cleaner.clean_dataframe(newspaper.copy(), ['headline', 'snippet', 'article'])

cleaning data headline:   0%|          | 0/1525 [00:00<?, ?it/s]

cleaning data snippet:   0%|          | 0/1525 [00:00<?, ?it/s]

cleaning data article:   0%|          | 0/1525 [00:00<?, ?it/s]

Successfully cleaned data frame.


In [911]:
newspaper_topicmodel = topicmodel_cleaner.clean_dataframe(newspaper.copy(), ['headline', 'snippet', 'article'])


cleaning data headline:   0%|          | 0/1525 [00:00<?, ?it/s]

cleaning data snippet:   0%|          | 0/1525 [00:00<?, ?it/s]

cleaning data article:   0%|          | 0/1525 [00:00<?, ?it/s]

Successfully cleaned data frame.


In [912]:
newspaper_sentiment

,publication_date,headline,snippet,article,Search Term,Source,Keywords
4,2017-09-24,tv young sheldon,monday sept good doctor return,eastern abc classic ballroom vietnam war weta ...,Young Thug,Washington Post,young thug
7,2022-01-14,watch friday netflix,chillin island hbo gunna love great crew,eastern shark tank abc share toast drag race v...,Gunna,Washington Post,gunna
10,2017-04-25,tv tale hulu,wednesday april rashida blackish,eastern survivor game friendship castaway catc...,Young Thug,Washington Post,young thug
13,2015-10-01,biii x x ink,abound option contest week ask,click skip new contest ask week week empress c...,YSL,Washington Post,sb
14,2022-05-10,tuesday quiet day nebraska west virginia young...,catch,day quieter,Young Thug,Washington Post,young thug
...,...,...,...,...,...,...,...
1023,None,rock love,rock love boat run money celeb cameo think,rock love boat run money celeb cameo think mer...,nan,New York Post,gunna
1026,None,christina hall ring solo spa outing news,hgtv star look diamond left ring finger place ...,christina spa nice ring christina hall ring hi...,YSL,New York Post,"ysl, sb"
1028,None,madonna crop art basel sex,girl book miami,express madonna guest honor saint laurent bash...,YSL,New York Post,ysl
1029,None,launch,right reason design,right reason design not sketch not drape say s...,YSL,New York Post,ysl


In [913]:
newspaper_topicmodel = newspaper_topicmodel.reset_index().drop('index', axis=1)

In [914]:
newspaper_sentiment = newspaper_sentiment.reset_index().drop('index', axis=1)

In [915]:
newspaper_sentiment.to_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/Cleaned Data/Sentiment Analysis/Newspaper/newspaper.csv')
newspaper_topicmodel.to_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/Cleaned Data/Topic Modeling/Newspaper/newspaper.csv')

# News Station

The news station we have collected data from:
    
   - WSB TV
   - GPB TV
   - Atlanta Fox 5
        - Headlines
        - Youtube comment 

## 2.1 WSB TV

In [916]:
from datetime import datetime
from dateutil.parser import parse


def parse_date(date_str):
    """
    Convert a date string into a proper datetime object.
    Handles both relative dates (e.g., '2 days ago') and absolute dates (e.g., 'April 20, 2018 3:34pm EDT').
    """
    try:
        # Handle relative dates
        if 'day' in date_str:
            days_ago = int(date_str.split()[0])  # Extract the number of days
            return datetime.now() - timedelta(days=days_ago)
        elif 'yesterday' in date_str.lower():
            return datetime.now() - timedelta(days=1)

        # Handle absolute dates
        return parse(date_str)
    except Exception as e:
        return None 


In [917]:
wsb = pd.read_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/UnCleaned Data/News Station/WSB_TV_data.csv')

# Apply the parsing function to the 'date' column
wsb['date'] = wsb['date'].apply(parse_date)

# Drop rows where the date could not be parsed
wsb = wsb.dropna(subset=['date'])

# Drop unnecessary columns and reorder
wsb.drop('link', axis=1, inplace=True)
wsb = wsb[['date', 'title']]

# Convert to just the date part (if needed)
wsb['date'] = wsb['date'].dt.date
wsb

,date,title
0,2024-11-27,YSL Trial: Jurors will continue deliberating a...
1,2024-12-03,YSL Trial: Jurors will continue deliberating a...
2,2024-12-03,YSL Trial: Jurors will continue deliberating a...
3,2024-12-27,YSL Trial: Jurors will continue deliberating a...
4,2024-12-04,"YSL Trial: Yak Gotti found not guilty, Shannon..."
...,...,...
185,2024-10-08,"Co-founder says YSL is a music label, believes..."
186,2024-02-29,"Co-founder says YSL is a music label, believes..."
187,2024-06-10,"Co-founder says YSL is a music label, believes..."
188,2024-02-23,Hundreds of potential jurors questioned in You...


In [918]:
wsb['Keywords'] = wsb['title'].apply(lambda title: find_matching_keywords(title, keywords))

In [919]:
wsb.dropna(subset=['Keywords'], inplace=True)

In [920]:
wsb

,date,title,Keywords
0,2024-11-27,YSL Trial: Jurors will continue deliberating a...,ysl
1,2024-12-03,YSL Trial: Jurors will continue deliberating a...,ysl
2,2024-12-03,YSL Trial: Jurors will continue deliberating a...,ysl
3,2024-12-27,YSL Trial: Jurors will continue deliberating a...,ysl
4,2024-12-04,"YSL Trial: Yak Gotti found not guilty, Shannon...","ysl, yak gotti, yak, gotti, shannon stillwell,..."
...,...,...,...
185,2024-10-08,"Co-founder says YSL is a music label, believes...",ysl
186,2024-02-29,"Co-founder says YSL is a music label, believes...",ysl
187,2024-06-10,"Co-founder says YSL is a music label, believes...",ysl
188,2024-02-23,Hundreds of potential jurors questioned in You...,"young thug, ysl"


In [921]:
wsb['title'] = wsb['title'].astype('str')

In [922]:
wsb_sentiment = sentiment_cleaner.clean_dataframe(wsb.copy(), ['title'])
wsb_topicmodel  = topicmodel_cleaner.clean_dataframe(wsb.copy(), ['title'])

cleaning data title:   0%|          | 0/88 [00:00<?, ?it/s]

Successfully cleaned data frame.


cleaning data title:   0%|          | 0/88 [00:00<?, ?it/s]

Successfully cleaned data frame.


In [923]:
wsb_topicmodel.to_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/Cleaned Data/Topic Modeling/News Station/wsb.csv')
wsb_sentiment.to_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/Cleaned Data/Sentiment Analysis/News Station/wsb.csv')

## 2.2 GPB TV

In [924]:
gpb = pd.read_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/UnCleaned Data/News Station/gpb.csv')
gpb

,Unnamed: 0,headlines,publication_date,snippets,links,article
0,1,Why the judge in Young Thug’s trial was ...,"July 17, 2024",Fulton County Chief Judge Ural Glanville was t...,https://www.gpb.org/news/2024/07/17/why-the-ju...,\n \n\n\n \n\n\n\nCaption\...
1,2,Defense attorney for rapper Young Thug f...,"June 11, 2024",A Georgia judge has ordered a lawyer for rappe...,https://www.gpb.org/news/2024/06/11/defense-at...,NaN
2,3,Last 2 defendants not guilty of murder i...,"December 03, 2024",The long-running gang and racketeering trial i...,https://www.gpb.org/news/2024/12/03/last-2-def...,NaN
3,4,Georgia Today: Last day of early voting;...,"November 01, 2024","On the Friday, Nov.",https://www.gpb.org/news/2024/11/01/georgia-to...,NaN
4,5,Georgia Today: Service for Rosalynn Cart...,"November 27, 2023","On the Monday, Nov.",https://www.gpb.org/news/2023/11/27/georgia-to...,NaN
...,...,...,...,...,...,...
117,118,Historian Uncovers The Racist Roots Of T...,"June 02, 2021",Carol Anderson says the Second Amendment was d...,https://www.gpb.org/news/2021/06/02/historian-...,\n Carol Anderson says the Second Amend...
118,119,"Jon Batiste, Justin Bieber and Billie Ei...","November 23, 2021",Pianist and composer Jon Batiste was nominated...,https://www.gpb.org/news/2021/11/23/jon-batist...,\n \n\n\n \n\n\n\nCaption\...
119,120,2022 Grammy Awards: The full list of nom...,"April 04, 2022",Every nominee and winner from all 86 categorie...,https://www.gpb.org/news/2022/04/04/2022-gramm...,\n \n\n\n \n\n\n\nCaption\...
120,121,Political Rewind: Fani Willis considers ...,"March 21, 2023",Tuesday on Political Rewind: Trump's lawyers f...,https://www.gpb.org/news/2023/03/21/political-...,NaN


In [925]:
gpb.drop('Unnamed: 0', axis=1, inplace=True)
gpb.drop('links', axis=1, inplace=True)
gpb['publication_date'] = gpb['publication_date'].apply(parse_date)
gpb['headlines'] = gpb['headlines'].astype('str')
gpb['snippets'] = gpb['snippets'].astype('str')
gpb['article'] = gpb['article'].astype('str')
gpb['complete'] = gpb['headlines']+' '+gpb['snippets']+' '+gpb['article']
gpb['Keywords'] = gpb['complete'].apply(lambda article: find_matching_keywords(article, keywords))
gpb.drop('complete', axis=1, inplace=True)
gpb.dropna(subset=['Keywords'], inplace=True)
gpb = gpb[['publication_date', 'headlines', 'snippets', 'article', 'Keywords']]

In [926]:
gpb_sentiment = sentiment_cleaner.clean_dataframe(gpb.copy(), ['headlines', 'snippets', 'article'])
gpb_topicmodel = topicmodel_cleaner.clean_dataframe(gpb.copy(), ['headlines', 'snippets', 'article'])

cleaning data headlines:   0%|          | 0/55 [00:00<?, ?it/s]

cleaning data snippets:   0%|          | 0/55 [00:00<?, ?it/s]

cleaning data article:   0%|          | 0/55 [00:00<?, ?it/s]

Successfully cleaned data frame.


cleaning data headlines:   0%|          | 0/55 [00:00<?, ?it/s]

cleaning data snippets:   0%|          | 0/55 [00:00<?, ?it/s]

cleaning data article:   0%|          | 0/55 [00:00<?, ?it/s]

Successfully cleaned data frame.


In [927]:
gpb_topicmodel

,publication_date,headlines,snippets,article,Keywords
0,2024-07-17,judge young thugs trial recused case,fulton county chief judge ural glanville taken...,caption young thug performs halftime game atla...,"young thug, ysl, ysl rico, rico, jeffrey willi..."
1,2024-06-11,defense attorney rapper young thug found conte...,georgia judge ordered lawyer rapper young thug...,nan,young thug
2,2024-12-03,defendants guilty murder gang trial led rapper...,longrunning gang racketeering trial atlanta le...,nan,young thug
3,2024-11-01,georgia today day early voting dock collapse w...,friday nov,nan,young thug
4,2023-11-27,georgia today service rosalynn carter underway...,monday nov,nan,young thug
5,2023-11-09,georgia today actors strike ends young thug ly...,thursday nov edition georgia today actors stri...,nan,young thug
6,2022-05-11,rapper gunna booked jail racketeering charge,rapper gunna booked jail atlanta wednesday rac...,nan,"young thug, gunna"
8,2022-12-15,rapper gunna pleads guilty racketeering case a...,rapper gunna pleaded guilty atlanta racketeeri...,nan,"young thug, gunna"
9,2022-07-07,rapper gunna denied bond gang racketeering case,judge atlanta denied bond rapper gunna s charg...,nan,"young thug, gunna"
10,2024-09-05,rich homie quan atlanta rapper known trap jams...,rich homie quan died atlanta rapper gained mai...,nan,young thug


In [928]:
gpb_sentiment.to_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/Cleaned Data/Sentiment Analysis/News Station/gpb.csv')
gpb_topicmodel.to_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/Cleaned Data/Topic Modeling/News Station/gpb.csv')

## 2.3 Atlanta Fox5

In [929]:
fox5 = pd.read_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/UnCleaned Data/News Station/atlanta_fox5csv')
fox5.drop('Unnamed: 0', axis=1, inplace=True)
fox5.drop('links', axis=1, inplace=True)
fox5

,headlines,publication_date,snippets,article
0,YSL RICO case: Rapper Yak Gotti to remain in j...,"December 17, 2024 7:18pm EST",Two co-defendants in the YSL RICO trial appear...,Co-defendants in YSL RICO trial appear in c...
1,Co-defendants in YSL RICO trial appear in court,"December 17, 2024 5:29pm EST",One of the defendants in the YSL RICO trial is...,NaN
2,Deamonte Kendrick and Shannon Stillwell in court,"December 17, 2024 12:16pm EST",A little more than a month after rapper Young ...,NaN
3,YSL RICO trial defendant files class-action la...,"December 12, 2024 4:43pm EST",A Suwanee man who exploited people recovering ...,article Fulton County Jail (FOX 5) The Brief...
4,Young Thug allowed to visit Atlanta home start...,"December 11, 2024 5:37pm EST","Attorneys for Deamonte Kendrick, who goes by t...",Young Thug gets Atlanta ban terms amended J...
...,...,...,...,...
1304,Air force bids military dog heartwarming final...,"February 7, 2018 8:57pm EST",Military working dogs and their handlers stood...,"Image 1 of 8 ▼ DOVER AIR FORCE BASE, De..."
1305,House passes temporary federal spending bill t...,"February 6, 2018 11:24pm EST",Buoyed by the sudden likelihood of a budget pa...,article Photo by Al Drago-Pool/Getty Images ...
1306,Southwest Airlines flies 62 dogs and cats out ...,"January 29, 2018 7:44pm EST",(FOX NEWS) - Talk about some lucky dogs.,Image 1 of 12 ▼ (Shelley Castle Photography...
1307,Hurricane relief effort led by ex-US president...,"January 24, 2018 3:38pm EST",A hurricane relief effort led by five former U...,article Former United States Presidents Jimmy ...


In [930]:
fox5['publication_date'] = fox5['publication_date'].apply(parse_date)
fox5['publication_date'] = pd.to_datetime(fox5['publication_date']).dt.date

In [931]:
fox5['headlines'] = fox5['headlines'].astype('str')
fox5['snippets'] = fox5['snippets'].astype('str')
fox5['article'] = fox5['article'].astype('str')
fox5['complete'] = fox5['headlines'] + ' ' + fox5['snippets'] + ' ' + fox5['article']
fox5['Keywords'] = fox5['headlines'].apply(lambda x: find_matching_keywords(x, keywords))
fox5.drop('complete', axis=1, inplace=True)
fox5 = fox5[['publication_date', 'headlines', 'snippets', 'article', 'Keywords']].copy()
fox5

,publication_date,headlines,snippets,article,Keywords
0,2024-12-17,YSL RICO case: Rapper Yak Gotti to remain in j...,Two co-defendants in the YSL RICO trial appear...,Co-defendants in YSL RICO trial appear in c...,"ysl, ysl rico, rico, yak gotti, yak, gotti, sh..."
1,2024-12-17,Co-defendants in YSL RICO trial appear in court,One of the defendants in the YSL RICO trial is...,nan,"ysl, ysl rico, rico"
2,2024-12-17,Deamonte Kendrick and Shannon Stillwell in court,A little more than a month after rapper Young ...,nan,"shannon stillwell, shannon, stillwell, deamont..."
3,2024-12-12,YSL RICO trial defendant files class-action la...,A Suwanee man who exploited people recovering ...,article Fulton County Jail (FOX 5) The Brief...,"ysl, ysl rico, rico"
4,2024-12-11,Young Thug allowed to visit Atlanta home start...,"Attorneys for Deamonte Kendrick, who goes by t...",Young Thug gets Atlanta ban terms amended J...,young thug
...,...,...,...,...,...
1304,2018-02-07,Air force bids military dog heartwarming final...,Military working dogs and their handlers stood...,"Image 1 of 8 ▼ DOVER AIR FORCE BASE, De...",None
1305,2018-02-06,House passes temporary federal spending bill t...,Buoyed by the sudden likelihood of a budget pa...,article Photo by Al Drago-Pool/Getty Images ...,None
1306,2018-01-29,Southwest Airlines flies 62 dogs and cats out ...,(FOX NEWS) - Talk about some lucky dogs.,Image 1 of 12 ▼ (Shelley Castle Photography...,rico
1307,2018-01-24,Hurricane relief effort led by ex-US president...,A hurricane relief effort led by five former U...,article Former United States Presidents Jimmy ...,None


In [932]:
fox5.dropna(subset=['Keywords'], inplace=True)
fox5

,publication_date,headlines,snippets,article,Keywords
0,2024-12-17,YSL RICO case: Rapper Yak Gotti to remain in j...,Two co-defendants in the YSL RICO trial appear...,Co-defendants in YSL RICO trial appear in c...,"ysl, ysl rico, rico, yak gotti, yak, gotti, sh..."
1,2024-12-17,Co-defendants in YSL RICO trial appear in court,One of the defendants in the YSL RICO trial is...,nan,"ysl, ysl rico, rico"
2,2024-12-17,Deamonte Kendrick and Shannon Stillwell in court,A little more than a month after rapper Young ...,nan,"shannon stillwell, shannon, stillwell, deamont..."
3,2024-12-12,YSL RICO trial defendant files class-action la...,A Suwanee man who exploited people recovering ...,article Fulton County Jail (FOX 5) The Brief...,"ysl, ysl rico, rico"
4,2024-12-11,Young Thug allowed to visit Atlanta home start...,"Attorneys for Deamonte Kendrick, who goes by t...",Young Thug gets Atlanta ban terms amended J...,young thug
...,...,...,...,...,...
1286,2018-05-07,Puerto Rico mourns loss of 9 airmen killed in ...,"The nine men came from across the island, join...","Image 1 of 3 ▼ First Lt. David Albandoz, Se...",rico
1292,2018-04-18,Excavator blamed for island-wide blackout in P...,An island-wide blackout hit Puerto Rico on Wed...,"article damian entwistle / Flickr SAN JUAN,...",rico
1300,2018-03-05,"Huge waves slam into Puerto Rico, force evacua...",Waves nearly 30 feet high from a U.S. winter ...,article (Photo courtesy U.S. Coast Guard) S...,rico
1306,2018-01-29,Southwest Airlines flies 62 dogs and cats out ...,(FOX NEWS) - Talk about some lucky dogs.,Image 1 of 12 ▼ (Shelley Castle Photography...,rico


In [933]:
fox5_sentiment = sentiment_cleaner.clean_dataframe(fox5.copy(), ['headlines', 'snippets', 'article'])
fox5_topicmodels = topicmodel_cleaner.clean_dataframe(fox5.copy(), ['headlines', 'snippets', 'article'])

cleaning data headlines:   0%|          | 0/511 [00:00<?, ?it/s]

cleaning data snippets:   0%|          | 0/511 [00:00<?, ?it/s]

cleaning data article:   0%|          | 0/511 [00:00<?, ?it/s]

Successfully cleaned data frame.


cleaning data headlines:   0%|          | 0/511 [00:00<?, ?it/s]

cleaning data snippets:   0%|          | 0/511 [00:00<?, ?it/s]

cleaning data article:   0%|          | 0/511 [00:00<?, ?it/s]

Successfully cleaned data frame.


In [934]:
fox5_topicmodels.to_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/Cleaned Data/Topic Modeling/News Station/fox5.csv')
fox5_sentiment.to_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/Cleaned Data/Sentiment Analysis/News Station/fox5.csv')

### 2.3.1 Atlanta Fox5 YouTube

In [935]:
fox5_youtube = pd.read_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/UnCleaned Data/News Station/Fox 5 atlanta Youtube/fox5_atlanta_complete.csv', low_memory=False)
fox5_youtube.rename(columns={'likes_x':'video likes', 'likes_y':'comment likes', 'comments': 'total comment'}, inplace=True)
fox5_youtube = fox5_youtube[['video_id', 'upload_date', 'title', 'views', 'video likes', 'total comment', 'published_at', 'comment', 'comment likes', 'replies']]
fox5_youtube['upload_date'] = fox5_youtube['upload_date'].apply(parse_date)
fox5_youtube['published_at'] = fox5_youtube['published_at'].apply(parse_date)
fox5_youtube['title'] = fox5_youtube['title'].astype('str')
fox5_youtube['views'] = fox5_youtube['views'].fillna(0).astype('int64')
fox5_youtube['total comment'] = fox5_youtube['total comment'].fillna(0).astype('int64')
fox5_youtube['comment likes'] = fox5_youtube['comment likes'].fillna(0).astype('int64')
fox5_youtube['replies'] = fox5_youtube['replies'].fillna(0).astype('int64')
fox5_youtube['comment'] = fox5_youtube['comment'].astype('str')
fox5_youtube

,video_id,upload_date,title,views,video likes,total comment,published_at,comment,comment likes,replies
0,rRPiY8MPcG4,2024-12-17 23:05:49+00:00,Co-defendants in YSL RICO trial appear in cour...,749,3,1,2024-12-19 10:07:44+00:00,Plea deals for murderers is insane.,0,0
1,VCdbAmV6Oa0,2024-12-17 20:41:46+00:00,Deamonte Kendrick and Shannon Stillwell in cou...,428,7,2,2024-12-17 22:45:17+00:00,Nice pink jacket to go to work in and then be...,0,0
2,VCdbAmV6Oa0,2024-12-17 20:41:46+00:00,Deamonte Kendrick and Shannon Stillwell in cou...,428,7,2,2024-12-17 22:33:59+00:00,Let Both Men Go Home!,0,0
3,XEEohTqbwUE,2024-12-12 20:35:13+00:00,WATCH LIVE: Amari Hall murder: Celeste Owens g...,14004,198,13,2024-12-17 21:29:13+00:00,😢,0,0
4,XEEohTqbwUE,2024-12-12 20:35:13+00:00,WATCH LIVE: Amari Hall murder: Celeste Owens g...,14004,198,13,2024-12-17 17:12:00+00:00,This precious little girl should have never ...,2,0
...,...,...,...,...,...,...,...,...,...,...
75605,QqiRmm8hvgc,2024-12-17 15:51:47+00:00,WATCH LIVE: Hearings for YSL defendants,8711,101,8,2024-12-17 19:56:12+00:00,He should have went to trial.,0,0
75606,QqiRmm8hvgc,2024-12-17 15:51:47+00:00,WATCH LIVE: Hearings for YSL defendants,8711,101,8,2024-12-17 19:27:27+00:00,Thankyou Fox 5 and also if you can get the bon...,0,0
75607,QqiRmm8hvgc,2024-12-17 15:51:47+00:00,WATCH LIVE: Hearings for YSL defendants,8711,101,8,2024-12-17 19:25:55+00:00,I wish them all well&amp; the prosecutors shou...,2,0
75608,QqiRmm8hvgc,2024-12-17 15:51:47+00:00,WATCH LIVE: Hearings for YSL defendants,8711,101,8,2024-12-17 19:07:01+00:00,Did he not know about the other trial?,0,0


In [936]:
fox5_youtube['complete'] = fox5_youtube['title'] +' '+fox5_youtube['comment']
fox5_youtube['Keywords'] = fox5_youtube['complete'].apply(lambda x: find_matching_keywords(x, keywords))
fox5_youtube.drop('complete', axis=1, inplace=True)

In [937]:
internet_sentiment_cleaner = TextCleaner(remove_stopwords=True, stemming=True, lemmatize=True, autocorrect= False, translation=False)

In [938]:
fox5_youtube_sentiment = internet_sentiment_cleaner.clean_dataframe(fox5_youtube.copy(), ['title', 'comment'])
fox5_youtube_topicmodels = topicmodel_cleaner.clean_dataframe(fox5_youtube.copy(), ['title', 'comment'])

cleaning data title:   0%|          | 0/75610 [00:00<?, ?it/s]

cleaning data comment:   0%|          | 0/75610 [00:00<?, ?it/s]

Successfully cleaned data frame.


cleaning data title:   0%|          | 0/75610 [00:00<?, ?it/s]

cleaning data comment:   0%|          | 0/75610 [00:00<?, ?it/s]

Successfully cleaned data frame.


In [939]:
fox5_youtube_sentiment.to_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/Cleaned Data/Sentiment Analysis/News Station/fox5 YouTube.csv')
fox5_youtube_topicmodels.to_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/Cleaned Data/Topic Modeling/News Station/fox5 YouTube.csv')

# 3 Magazine

## 3.1 People's Magazine

In [940]:
def extract_date(date_str):
    """
    Extract the date from a string containing a date and time.
    Handles formats like 'Updated on July 15, 2024 04:35PM EDT'.
    """
    try:
        # Remove prefixes like 'Updated on' or 'Published on'
        if 'Updated on' in date_str:
            date_str = date_str.replace('Updated on', '').strip()
        elif 'Published on' in date_str:
            date_str = date_str.replace('Published on', '').strip()
        elif 'Published: ' in date_str:
            date_str = date_str.replace('Published: ', '').strip()
        elif 'Updated: ' in date_str:
            date_str = date_str.replace('Updated: ', '').strip()
        
        # Parse the date string
        parsed_date = parse(date_str)
        return parsed_date
    except Exception as e:
        return None

In [941]:
people = pd.read_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/UnCleaned Data/Magazine/people_magazine.csv', index_col=0)
people

,headlines,links,article,pub_date
1,What to Know About the High-Profile Young Thug...,https://people.com/young-thug-ysl-rico-trail-w...,\n\n\n\n\n \nYoung Thug.\nPhoto: Prince Willia...,"Updated on July 15, 2024 04:35PM EDT"
2,What to Know About the High-Profile Young Thug...,https://people.com/young-thug-ysl-rico-trail-w...,The high-profile trial involving rapper Young...,"Updated on July 15, 2024 04:35PM EDT"
3,Who Is Young Thug's Girlfriend? All About Mari...,https://people.com/who-is-mariah-the-scientist...,\n\n\n\n\n \nYoung Thug speaks onstage at the ...,"Published on November 1, 2024 05:26PM EDT"
4,Who Is Young Thug's Girlfriend? All About Mari...,https://people.com/who-is-mariah-the-scientist...,Young Thug has a special duet partner in his ...,"Published on November 1, 2024 05:26PM EDT"
5,Young Thug Pleads Guilty to Criminal Activity ...,https://people.com/young-thug-pleads-guilty-in...,\n\n\n\n\n \nYoung Thug in 2021.\nPhoto: Paras...,"Published on October 31, 2024 07:47PM EDT"
...,...,...,...,...
236,New England Patriots' Danny Amendola Confirms ...,https://people.com/style/danny-amendola-signs-...,Most people are probably familiar with Danny ...,"Published on July 13, 2017 11:46AM EDT"
237,'SNL' Sets First 2022 Show with Ariana DeBose ...,https://people.com/tv/snl-sets-first-2022-show...,\n\n\n\n\n \nAriana DeBose and Roddy Ricch.\nP...,"Published on January 6, 2022 05:21PM EST"
238,'SNL' Sets First 2022 Show with Ariana DeBose ...,https://people.com/tv/snl-sets-first-2022-show...,Saturday Night Live is back for more in 2022!...,"Published on January 6, 2022 05:21PM EST"
239,BET Hip Hop Awards 2020: Everything to Know Ab...,https://people.com/music/bet-hip-hop-awards-20...,"\n\n\n\n\n \nRoddy Ricch, Megan Thee Stallion,...","Published on October 27, 2020 07:56PM EDT"


In [942]:
people.drop('links', axis=1, inplace=True)
people.rename(columns={'pub_date': 'publication_date'}, inplace=True)
people['headlines'] = people['headlines'].astype('str')
people['article'] = people['article'].astype('str')
people['complete'] = people['headlines'] + ' ' + people['article']
people['Keywords'] = people['complete'].apply(lambda x: find_matching_keywords(x, keywords))
people.drop('complete', axis=1, inplace=True)
people['publication_date'] = pd.to_datetime(people['publication_date'].apply(extract_date)).dt.date
people

,headlines,article,publication_date,Keywords
1,What to Know About the High-Profile Young Thug...,\n\n\n\n\n \nYoung Thug.\nPhoto: Prince Willia...,2024-07-15,"young thug, gunna, ysl, ysl rico, rico, jeffre..."
2,What to Know About the High-Profile Young Thug...,The high-profile trial involving rapper Young...,2024-07-15,"young thug, gunna, ysl, ysl rico, rico, jeffre..."
3,Who Is Young Thug's Girlfriend? All About Mari...,\n\n\n\n\n \nYoung Thug speaks onstage at the ...,2024-11-01,"young thug, rico, sb, williams"
4,Who Is Young Thug's Girlfriend? All About Mari...,Young Thug has a special duet partner in his ...,2024-11-01,"young thug, rico, sb, williams"
5,Young Thug Pleads Guilty to Criminal Activity ...,\n\n\n\n\n \nYoung Thug in 2021.\nPhoto: Paras...,2024-10-31,"young thug, gunna, ysl, rico, sb, williams, ro..."
...,...,...,...,...
236,New England Patriots' Danny Amendola Confirms ...,Most people are probably familiar with Danny ...,2017-07-13,young thug
237,'SNL' Sets First 2022 Show with Ariana DeBose ...,\n\n\n\n\n \nAriana DeBose and Roddy Ricch.\nP...,2022-01-06,young thug
238,'SNL' Sets First 2022 Show with Ariana DeBose ...,Saturday Night Live is back for more in 2022!...,2022-01-06,young thug
239,BET Hip Hop Awards 2020: Everything to Know Ab...,"\n\n\n\n\n \nRoddy Ricch, Megan Thee Stallion,...",2020-10-27,"young thug, gunna"


In [943]:
people.dropna(subset=['Keywords'], inplace=True)
people = people[['publication_date', 'headlines', 'article', 'Keywords']]
people

,publication_date,headlines,article,Keywords
1,2024-07-15,What to Know About the High-Profile Young Thug...,\n\n\n\n\n \nYoung Thug.\nPhoto: Prince Willia...,"young thug, gunna, ysl, ysl rico, rico, jeffre..."
2,2024-07-15,What to Know About the High-Profile Young Thug...,The high-profile trial involving rapper Young...,"young thug, gunna, ysl, ysl rico, rico, jeffre..."
3,2024-11-01,Who Is Young Thug's Girlfriend? All About Mari...,\n\n\n\n\n \nYoung Thug speaks onstage at the ...,"young thug, rico, sb, williams"
4,2024-11-01,Who Is Young Thug's Girlfriend? All About Mari...,Young Thug has a special duet partner in his ...,"young thug, rico, sb, williams"
5,2024-10-31,Young Thug Pleads Guilty to Criminal Activity ...,\n\n\n\n\n \nYoung Thug in 2021.\nPhoto: Paras...,"young thug, gunna, ysl, rico, sb, williams, ro..."
...,...,...,...,...
236,2017-07-13,New England Patriots' Danny Amendola Confirms ...,Most people are probably familiar with Danny ...,young thug
237,2022-01-06,'SNL' Sets First 2022 Show with Ariana DeBose ...,\n\n\n\n\n \nAriana DeBose and Roddy Ricch.\nP...,young thug
238,2022-01-06,'SNL' Sets First 2022 Show with Ariana DeBose ...,Saturday Night Live is back for more in 2022!...,young thug
239,2020-10-27,BET Hip Hop Awards 2020: Everything to Know Ab...,"\n\n\n\n\n \nRoddy Ricch, Megan Thee Stallion,...","young thug, gunna"


In [944]:
people_sentiment = sentiment_cleaner.clean_dataframe(people.copy(), ['headlines', 'article'])
people_topicmodels = topicmodel_cleaner.clean_dataframe(people.copy(), ['headlines', 'article'])

cleaning data headlines:   0%|          | 0/234 [00:00<?, ?it/s]

cleaning data article:   0%|          | 0/234 [00:00<?, ?it/s]

Successfully cleaned data frame.


cleaning data headlines:   0%|          | 0/234 [00:00<?, ?it/s]

cleaning data article:   0%|          | 0/234 [00:00<?, ?it/s]

Successfully cleaned data frame.


In [945]:
people_topicmodels.to_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/Cleaned Data/Topic Modeling/Magazine/People.csv')
people_sentiment.to_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/Cleaned Data/Sentiment Analysis/Magazine/People.csv')

## 3.2 Rolling Stone

In [946]:
rolling_stone = pd.read_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/UnCleaned Data/Magazine/rollingstones.csv')
rolling_stone.drop('links', axis=1, inplace=True)
rolling_stone

,article,Headline,Date
0,"\n\tWhile on stage in Lagos, Nigeria for the f...",Watch Tyla Join Gunna for ‘Jump’ at Blockbuste...,"Dec 23, 2024"
1,"\n\tWhile on stage in Lagos, Nigeria for the f...",Watch Tyla Join Gunna for ‘Jump’ at Blockbuste...,"Dec 23, 2024"
2,"\n\tWhile on stage in Lagos, Nigeria for the f...",Watch Tyla Join Gunna for ‘Jump’ at Blockbuste...,"Dec 23, 2024"
3,\n\tSince Kendrick Lamar took aim at Drake wit...,Everything That’s Happened in the Drake-Kendri...,"Dec 4, 2024"
4,\n\tSince Kendrick Lamar took aim at Drake wit...,Everything That’s Happened in the Drake-Kendri...,"Dec 4, 2024"
...,...,...,...
1155,Ask a casual fan what they like about Travis S...,The Stakes for Travis Scott Are Higher Than Ev...,"Aug 2, 2018"
1156,"Cardi B, the Carters (Beyoncé and Jay-Z), Chil...","VMAs 2018: Cardi B, the Carters, Childish Gamb...","Jul 16, 2018"
1157,"Cardi B, the Carters (Beyoncé and Jay-Z), Chil...","VMAs 2018: Cardi B, the Carters, Childish Gamb...","Jul 16, 2018"
1158,There’s a streak of fierce independence in Tay...,Taylor Bennett Is Getting Comfortable Being Hi...,"Jul 12, 2018"


In [947]:
rolling_stone['article'] = rolling_stone['article'].astype('str')
rolling_stone.rename(columns={'Date': 'publication_date','Headline': 'headlines'}, inplace=True)
rolling_stone['headlines'] = rolling_stone['headlines'].astype('str')
rolling_stone['publication_date'] = pd.to_datetime(rolling_stone['publication_date']).dt.date
rolling_stone['complete'] = rolling_stone['headlines'] + ' ' + rolling_stone['article']
rolling_stone['Keywords'] = rolling_stone['complete'].apply(lambda x: find_matching_keywords(x, keywords))
rolling_stone.drop('complete', axis=1, inplace=True)
rolling_stone = rolling_stone[['publication_date', 'headlines', 'article', 'Keywords']]
rolling_stone

,publication_date,headlines,article,Keywords
0,2024-12-23,Watch Tyla Join Gunna for ‘Jump’ at Blockbuste...,"\n\tWhile on stage in Lagos, Nigeria for the f...","young thug, gunna, sergio kitchens, sergio, ki..."
1,2024-12-23,Watch Tyla Join Gunna for ‘Jump’ at Blockbuste...,"\n\tWhile on stage in Lagos, Nigeria for the f...","young thug, gunna, sergio kitchens, sergio, ki..."
2,2024-12-23,Watch Tyla Join Gunna for ‘Jump’ at Blockbuste...,"\n\tWhile on stage in Lagos, Nigeria for the f...","young thug, gunna, sergio kitchens, sergio, ki..."
3,2024-12-04,Everything That’s Happened in the Drake-Kendri...,\n\tSince Kendrick Lamar took aim at Drake wit...,"young thug, rico, williams, kendrick"
4,2024-12-04,Everything That’s Happened in the Drake-Kendri...,\n\tSince Kendrick Lamar took aim at Drake wit...,"young thug, rico, williams, kendrick"
...,...,...,...,...
1155,2018-08-02,The Stakes for Travis Scott Are Higher Than Ev...,Ask a casual fan what they like about Travis S...,young thug
1156,2018-07-16,"VMAs 2018: Cardi B, the Carters, Childish Gamb...","Cardi B, the Carters (Beyoncé and Jay-Z), Chil...","young thug, sb, kendrick"
1157,2018-07-16,"VMAs 2018: Cardi B, the Carters, Childish Gamb...","Cardi B, the Carters (Beyoncé and Jay-Z), Chil...","young thug, sb, kendrick"
1158,2018-07-12,Taylor Bennett Is Getting Comfortable Being Hi...,There’s a streak of fierce independence in Tay...,young thug


In [948]:
rolling_stone = rolling_stone.dropna(subset=['Keywords'])
rolling_stone

,publication_date,headlines,article,Keywords
0,2024-12-23,Watch Tyla Join Gunna for ‘Jump’ at Blockbuste...,"\n\tWhile on stage in Lagos, Nigeria for the f...","young thug, gunna, sergio kitchens, sergio, ki..."
1,2024-12-23,Watch Tyla Join Gunna for ‘Jump’ at Blockbuste...,"\n\tWhile on stage in Lagos, Nigeria for the f...","young thug, gunna, sergio kitchens, sergio, ki..."
2,2024-12-23,Watch Tyla Join Gunna for ‘Jump’ at Blockbuste...,"\n\tWhile on stage in Lagos, Nigeria for the f...","young thug, gunna, sergio kitchens, sergio, ki..."
3,2024-12-04,Everything That’s Happened in the Drake-Kendri...,\n\tSince Kendrick Lamar took aim at Drake wit...,"young thug, rico, williams, kendrick"
4,2024-12-04,Everything That’s Happened in the Drake-Kendri...,\n\tSince Kendrick Lamar took aim at Drake wit...,"young thug, rico, williams, kendrick"
...,...,...,...,...
1155,2018-08-02,The Stakes for Travis Scott Are Higher Than Ev...,Ask a casual fan what they like about Travis S...,young thug
1156,2018-07-16,"VMAs 2018: Cardi B, the Carters, Childish Gamb...","Cardi B, the Carters (Beyoncé and Jay-Z), Chil...","young thug, sb, kendrick"
1157,2018-07-16,"VMAs 2018: Cardi B, the Carters, Childish Gamb...","Cardi B, the Carters (Beyoncé and Jay-Z), Chil...","young thug, sb, kendrick"
1158,2018-07-12,Taylor Bennett Is Getting Comfortable Being Hi...,There’s a streak of fierce independence in Tay...,young thug


In [949]:
rollingstone_sentiment = sentiment_cleaner.clean_dataframe(rolling_stone.copy(), ['headlines', 'article'])
rollingstone_topicmodels = topicmodel_cleaner.clean_dataframe(rolling_stone.copy(), ['headlines', 'article'])

cleaning data headlines:   0%|          | 0/1031 [00:00<?, ?it/s]

cleaning data article:   0%|          | 0/1031 [00:00<?, ?it/s]

Successfully cleaned data frame.


cleaning data headlines:   0%|          | 0/1031 [00:00<?, ?it/s]

cleaning data article:   0%|          | 0/1031 [00:00<?, ?it/s]

Successfully cleaned data frame.


In [950]:
rollingstone_sentiment.to_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/Cleaned Data/Sentiment Analysis/Magazine/rollingstone.csv')
rollingstone_topicmodels.to_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/Cleaned Data/Topic Modeling/Magazine/rollingstone.csv')

## 3.3 Shade Room

In [951]:
shaderoom = pd.read_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/UnCleaned Data/Magazine/shade_room.csv')
shaderoom.drop('embedded_url', axis=1, inplace=True)
shaderoom['date'] = pd.to_datetime(shaderoom['date']).dt.date
shaderoom.rename(columns={'date': 'publication_date', 'headline':'headlines'}, inplace=True)
shaderoom['headlines'] = shaderoom['headlines'].astype('str')
shaderoom['article'] = shaderoom['article'].astype('str')
shaderoom['complete'] = shaderoom['headlines'] + ' ' + shaderoom['article']
shaderoom['Keywords'] = shaderoom['complete'].apply(lambda x: find_matching_keywords(x, keywords))
shaderoom.drop('complete', axis=1, inplace=True)
shaderoom = shaderoom[['publication_date', 'headlines', 'article', 'Keywords']]
shaderoom

,publication_date,headlines,article,Keywords
0,2024-11-23,"C’mon, Princess Treatment! Mariah The Scientis...",\n\n\t\t\n\t\t\t\t\n\t\tIf MY MAN! MY MAN! MY ...,"young thug, ysl, ysl rico, rico, paige reese w..."
1,2024-11-10,Whew! Gunna’s Brother Weighs In After Young Th...,\n\n\t\t\n\t\t\t\t\n\t\tWhew! After Young Thug...,"young thug, gunna, ysl"
2,2024-11-09,Social Media Goes Off With Reactions After You...,"\n\n\t\t\n\t\t\t\t\n\t\tWhew, chile! The innan...","young thug, gunna, ysl, ysl rico, rico"
3,2024-11-04,Whew! Some Social Media Users Are Upset At Mar...,\n\n\t\t\n\t\t\t\t\n\t\tSome social media user...,young thug
4,2024-11-04,He’s Back! Young Thug Drops His First Few Mess...,\n\n\t\t\n\t\t\t\t\n\t\tYoung Thug has shared ...,"young thug, ysl, ysl rico, rico"
...,...,...,...,...
82,2020-12-28,Young Thug Seemingly Responds To Jerrika Karla...,\n\n\t\t\n\t\t\t\t\n\t\tSay it isn’t so! One o...,young thug
83,2015-03-25,"Roommate Talk: Dear TSR, Young Thug Almost Sto...",\n\n\t\t\n\t\t\t\t\n\t\t\nhttp://youtu.be/B94I...,young thug
84,NaT,"ROOMMATE TALK: “DEAR TSR, TI AND YOUNG THUG NE...",nan,young thug
85,2015-02-04,Young Thug Says He Would Never Buy A Jay-Z CD ...,\n\n\t\t\n\t\t\t\t\n\t\t\nYoung Thug recently ...,young thug


In [952]:
shaderoom.dropna(subset=['Keywords'], inplace=True)
shaderoom

,publication_date,headlines,article,Keywords
0,2024-11-23,"C’mon, Princess Treatment! Mariah The Scientis...",\n\n\t\t\n\t\t\t\t\n\t\tIf MY MAN! MY MAN! MY ...,"young thug, ysl, ysl rico, rico, paige reese w..."
1,2024-11-10,Whew! Gunna’s Brother Weighs In After Young Th...,\n\n\t\t\n\t\t\t\t\n\t\tWhew! After Young Thug...,"young thug, gunna, ysl"
2,2024-11-09,Social Media Goes Off With Reactions After You...,"\n\n\t\t\n\t\t\t\t\n\t\tWhew, chile! The innan...","young thug, gunna, ysl, ysl rico, rico"
3,2024-11-04,Whew! Some Social Media Users Are Upset At Mar...,\n\n\t\t\n\t\t\t\t\n\t\tSome social media user...,young thug
4,2024-11-04,He’s Back! Young Thug Drops His First Few Mess...,\n\n\t\t\n\t\t\t\t\n\t\tYoung Thug has shared ...,"young thug, ysl, ysl rico, rico"
...,...,...,...,...
81,2022-03-18,"LaKevia Jackson, Mother To Young Thug’s Son, F...",\n\n\t\t\n\t\t\t\t\n\t\tIt’s a scary time when...,"young thug, rico, sb, williams"
82,2020-12-28,Young Thug Seemingly Responds To Jerrika Karla...,\n\n\t\t\n\t\t\t\t\n\t\tSay it isn’t so! One o...,young thug
83,2015-03-25,"Roommate Talk: Dear TSR, Young Thug Almost Sto...",\n\n\t\t\n\t\t\t\t\n\t\t\nhttp://youtu.be/B94I...,young thug
84,NaT,"ROOMMATE TALK: “DEAR TSR, TI AND YOUNG THUG NE...",nan,young thug


In [953]:
shaderoom_sentiment = sentiment_cleaner.clean_dataframe(shaderoom.copy(), ['headlines', 'article'])
shaderoom_topicmodels = topicmodel_cleaner.clean_dataframe(shaderoom.copy(), ['headlines', 'article'])

cleaning data headlines:   0%|          | 0/85 [00:00<?, ?it/s]

cleaning data article:   0%|          | 0/85 [00:00<?, ?it/s]

Successfully cleaned data frame.


cleaning data headlines:   0%|          | 0/85 [00:00<?, ?it/s]

cleaning data article:   0%|          | 0/85 [00:00<?, ?it/s]

Successfully cleaned data frame.


In [954]:
shaderoom_topicmodels.to_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/Cleaned Data/Topic Modeling/Magazine/Shade Room.csv')
shaderoom_sentiment.to_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/Cleaned Data/Sentiment Analysis/Magazine/Shade Room.csv')

## 3.4 TIME

In [955]:
times = pd.read_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/UnCleaned Data/Magazine/Time_Magazine.csv', index_col=0)
times.rename(columns={'date': 'publication_date', 'headline': 'headlines'}, inplace=True)
times['publication_date'] = times['publication_date'].apply(extract_date)
times['publication_date'] = pd.to_datetime(times['publication_date'], utc=True).dt.date
times['headlines'] = times['headlines'].astype('str')
times['article'] = times['article'].astype('str')
times['complete'] = times['headlines'] + ' ' + times['article']
times.drop('link', axis=1, inplace=True)
times['Keywords'] = times['complete'].apply(lambda x: find_matching_keywords(x, keywords))
times.drop('complete', axis=1, inplace=True)
times

,publication_date,article,headlines,Keywords
1,2024-11-23,\n\n\t\t\n\t\t\t\t\n\t\tIf MY MAN! MY MAN! MY ...,nan,"young thug, ysl, ysl rico, rico, paige reese w..."
2,2024-11-10,\n\n\t\t\n\t\t\t\t\n\t\tWhew! After Young Thug...,nan,"young thug, gunna, ysl"
3,2024-11-09,"\n\n\t\t\n\t\t\t\t\n\t\tWhew, chile! The innan...",nan,"young thug, gunna, ysl, ysl rico, rico"
4,2024-11-04,\n\n\t\t\n\t\t\t\t\n\t\tSome social media user...,nan,young thug
5,2024-11-04,\n\n\t\t\n\t\t\t\t\n\t\tYoung Thug has shared ...,nan,"young thug, ysl, ysl rico, rico"
...,...,...,...,...
180,NaN,nan,Rogue One Review: Star Wars Spinoff Gets the J...,None
181,NaN,nan,Denmark: What Transformed Copenhagen Gunman Fr...,None
182,NaN,nan,Heroes Are Only Human | TIME,None
183,NaN,nan,The Outsiders at 50: Read S.E. Hinton's Letter...,None


In [956]:
times.dropna(subset=['Keywords'], inplace=True)
times

,publication_date,article,headlines,Keywords
1,2024-11-23,\n\n\t\t\n\t\t\t\t\n\t\tIf MY MAN! MY MAN! MY ...,nan,"young thug, ysl, ysl rico, rico, paige reese w..."
2,2024-11-10,\n\n\t\t\n\t\t\t\t\n\t\tWhew! After Young Thug...,nan,"young thug, gunna, ysl"
3,2024-11-09,"\n\n\t\t\n\t\t\t\t\n\t\tWhew, chile! The innan...",nan,"young thug, gunna, ysl, ysl rico, rico"
4,2024-11-04,\n\n\t\t\n\t\t\t\t\n\t\tSome social media user...,nan,young thug
5,2024-11-04,\n\n\t\t\n\t\t\t\t\n\t\tYoung Thug has shared ...,nan,"young thug, ysl, ysl rico, rico"
...,...,...,...,...
83,2015-03-25,\n\n\t\t\n\t\t\t\t\n\t\t\nhttp://youtu.be/B94I...,nan,young thug
84,2015-02-04,\n\n\t\t\n\t\t\t\t\n\t\t\nYoung Thug recently ...,nan,young thug
87,NaT,nan,"Young Thug, Concerns Over Rap Lyrics as Eviden...",young thug
88,NaT,nan,Here's Where Young Thug's Trial Stands | TIME,young thug


In [957]:
times_sentiment = sentiment_cleaner.clean_dataframe(times.copy(), ['headlines', 'article'])
times_topicmodels = topicmodel_cleaner.clean_dataframe(times.copy(), ['headlines', 'article'])

cleaning data headlines:   0%|          | 0/84 [00:00<?, ?it/s]

cleaning data article:   0%|          | 0/84 [00:00<?, ?it/s]

Successfully cleaned data frame.


cleaning data headlines:   0%|          | 0/84 [00:00<?, ?it/s]

cleaning data article:   0%|          | 0/84 [00:00<?, ?it/s]

Successfully cleaned data frame.


In [958]:
times_sentiment.to_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/Cleaned Data/Sentiment Analysis/Magazine/time.csv')
times_topicmodels.to_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/Cleaned Data/Topic Modeling/Magazine/time.csv')

## 3.5 TMZ

In [959]:
tmz = pd.read_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/UnCleaned Data/Magazine/tmz.csv', index_col=0)
tmz.rename(columns={'pub_date': 'publication_date'}, inplace=True)
tmz.drop('links', axis=1, inplace=True)
tmz['publication_date'] = pd.to_datetime(tmz['publication_date'].apply(extract_date)).dt.date
tmz['complete'] = tmz['headlines'] + ' ' + tmz['article']
tmz['Keywords'] = tmz['complete'].apply(lambda x: find_matching_keywords(x, keywords))
tmz.drop('complete', axis=1, inplace=True)
tmz = tmz[['publication_date', 'headlines', 'article', 'Keywords']]
tmz

,publication_date,headlines,article,Keywords
1,2024-07-15,What to Know About the High-Profile Young Thug...,\n\n\n\n\n \nYoung Thug.\nPhoto: Prince Willia...,"young thug, gunna, ysl, ysl rico, rico, jeffre..."
2,2024-07-15,What to Know About the High-Profile Young Thug...,The high-profile trial involving rapper Young...,"young thug, gunna, ysl, ysl rico, rico, jeffre..."
3,2024-11-01,Who Is Young Thug's Girlfriend? All About Mari...,\n\n\n\n\n \nYoung Thug speaks onstage at the ...,"young thug, rico, sb, williams"
4,2024-11-01,Who Is Young Thug's Girlfriend? All About Mari...,Young Thug has a special duet partner in his ...,"young thug, rico, sb, williams"
5,2024-10-31,Young Thug Pleads Guilty to Criminal Activity ...,\n\n\n\n\n \nYoung Thug in 2021.\nPhoto: Paras...,"young thug, gunna, ysl, rico, sb, williams, ro..."
...,...,...,...,...
236,2017-07-13,New England Patriots' Danny Amendola Confirms ...,Most people are probably familiar with Danny ...,young thug
237,2022-01-06,'SNL' Sets First 2022 Show with Ariana DeBose ...,\n\n\n\n\n \nAriana DeBose and Roddy Ricch.\nP...,young thug
238,2022-01-06,'SNL' Sets First 2022 Show with Ariana DeBose ...,Saturday Night Live is back for more in 2022!...,young thug
239,2020-10-27,BET Hip Hop Awards 2020: Everything to Know Ab...,"\n\n\n\n\n \nRoddy Ricch, Megan Thee Stallion,...","young thug, gunna"


In [960]:
tmz.dropna(subset=['Keywords'], inplace=True)
tmz

,publication_date,headlines,article,Keywords
1,2024-07-15,What to Know About the High-Profile Young Thug...,\n\n\n\n\n \nYoung Thug.\nPhoto: Prince Willia...,"young thug, gunna, ysl, ysl rico, rico, jeffre..."
2,2024-07-15,What to Know About the High-Profile Young Thug...,The high-profile trial involving rapper Young...,"young thug, gunna, ysl, ysl rico, rico, jeffre..."
3,2024-11-01,Who Is Young Thug's Girlfriend? All About Mari...,\n\n\n\n\n \nYoung Thug speaks onstage at the ...,"young thug, rico, sb, williams"
4,2024-11-01,Who Is Young Thug's Girlfriend? All About Mari...,Young Thug has a special duet partner in his ...,"young thug, rico, sb, williams"
5,2024-10-31,Young Thug Pleads Guilty to Criminal Activity ...,\n\n\n\n\n \nYoung Thug in 2021.\nPhoto: Paras...,"young thug, gunna, ysl, rico, sb, williams, ro..."
...,...,...,...,...
236,2017-07-13,New England Patriots' Danny Amendola Confirms ...,Most people are probably familiar with Danny ...,young thug
237,2022-01-06,'SNL' Sets First 2022 Show with Ariana DeBose ...,\n\n\n\n\n \nAriana DeBose and Roddy Ricch.\nP...,young thug
238,2022-01-06,'SNL' Sets First 2022 Show with Ariana DeBose ...,Saturday Night Live is back for more in 2022!...,young thug
239,2020-10-27,BET Hip Hop Awards 2020: Everything to Know Ab...,"\n\n\n\n\n \nRoddy Ricch, Megan Thee Stallion,...","young thug, gunna"


In [961]:
tmz_sentiment = sentiment_cleaner.clean_dataframe(tmz.copy(), ['headlines', 'article'])
tmz_topicmodels = topicmodel_cleaner.clean_dataframe(tmz.copy(), ['headlines', 'article'])

cleaning data headlines:   0%|          | 0/234 [00:00<?, ?it/s]

cleaning data article:   0%|          | 0/234 [00:00<?, ?it/s]

Successfully cleaned data frame.


cleaning data headlines:   0%|          | 0/234 [00:00<?, ?it/s]

cleaning data article:   0%|          | 0/234 [00:00<?, ?it/s]

Successfully cleaned data frame.


In [962]:
tmz_sentiment.to_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/Cleaned Data/Sentiment Analysis/Magazine/tmz.csv')
tmz_topicmodels.to_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/Cleaned Data/Topic Modeling/Magazine/tmz.csv')

## 3.6 XXL

In [963]:
xxl = pd.read_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/UnCleaned Data/Magazine/xxl.csv', index_col=0)
xxl.rename(columns={'pub_date': 'publication_date'}, inplace=True)
xxl.drop('links', axis=1, inplace=True)
xxl['publication_date'] = xxl['publication_date'].apply(extract_date)
xxl['headlines'] = xxl['headlines'].astype('str')
xxl['snippet'] = xxl['snippet'].astype('str')
xxl['article'] = xxl['article'].astype('str')
xxl['complete'] = xxl['headlines'] + ' ' + xxl['snippet'] + ' ' + xxl['article']
xxl['Keywords'] = xxl['complete'].apply(lambda x: find_matching_keywords(x, keywords))
xxl.drop('complete', axis=1, inplace=True)
xxl = xxl[['publication_date', 'headlines', 'snippet', 'article', 'Keywords']]
xxl

,publication_date,headlines,snippet,article,Keywords
1,2024-12-04,Jurors Speak About Young Thug Trial for the Fi...,Jurors on the Young Thug YSL RICO trial are fi...,Jurors on the Young Thug YSL RICO trial are fi...,"young thug, ysl, ysl rico, rico, yak gotti, ya..."
2,2024-11-12,"Young Thug Hits the Studio With Lil Baby, Futu...",Read More: Hip-Hop Reacts to Young Thug Being ...,Cop Your XXL Merch NowYoung Thug is back in th...,"young thug, ysl, ysl rico, rico, brian steel"
3,2024-11-11,Did Young Thug Really Make Those Comments Abou...,Young Thug Tweet Puzzles Fans The current stat...,Cop Your XXL Merch NowYoung Thug's recent twee...,"young thug, gunna, ysl, ysl rico, rico"
4,2024-11-13,We’re So Confused About What’s Going on With Y...,The curious case of the status between Young T...,Cop Your XXL Merch NowThe curious case of the ...,"young thug, gunna, ysl, ysl rico, rico, williams"
5,2024-11-13,Joe Budden Thinks Young Thug Would Be a Fool t...,Read More: We're So Confused About What's Goin...,Cop Your XXL Merch NowJoe Budden insists it wo...,"young thug, gunna, ysl, ysl rico, rico"
...,...,...,...,...,...
1323,2020-11-27,Young Thug Says He’s Never Paid Attention to A...,One would think the eclectic styles of Young T...,One would think the eclectic styles of Young T...,young thug
1324,2022-01-07,Jim Jones’ Mom Responds to Jim Saying She Taug...,It's all about my son growing up and me as a y...,Jim Jones may have said he was joking when he ...,None
1325,2022-01-28,"NLE Choppa, Kyle, BlocBoy JB and More – New Pr...","received another release date for Jan. 21, com...","This week in hip-hop, both the older and young...",None
1326,2022-02-05,Hilarious Comments on Kanye West and Kim Karda...,looks like Kanye West feeling adamant that his...,It looks like Kanye West feeling adamant that ...,young thug


In [964]:
xxl.dropna(subset=['Keywords'], inplace=True)
xxl_sentiment = sentiment_cleaner.clean_dataframe(xxl.copy(), ['headlines', 'snippet', 'article'])
xxl_topicmodel = topicmodel_cleaner.clean_dataframe(xxl.copy(), ['headlines', 'snippet', 'article'])
xxl

cleaning data headlines:   0%|          | 0/779 [00:00<?, ?it/s]

cleaning data snippet:   0%|          | 0/779 [00:00<?, ?it/s]

cleaning data article:   0%|          | 0/779 [00:00<?, ?it/s]

Successfully cleaned data frame.


cleaning data headlines:   0%|          | 0/779 [00:00<?, ?it/s]

cleaning data snippet:   0%|          | 0/779 [00:00<?, ?it/s]

cleaning data article:   0%|          | 0/779 [00:00<?, ?it/s]

Successfully cleaned data frame.


,publication_date,headlines,snippet,article,Keywords
1,2024-12-04,Jurors Speak About Young Thug Trial for the Fi...,Jurors on the Young Thug YSL RICO trial are fi...,Jurors on the Young Thug YSL RICO trial are fi...,"young thug, ysl, ysl rico, rico, yak gotti, ya..."
2,2024-11-12,"Young Thug Hits the Studio With Lil Baby, Futu...",Read More: Hip-Hop Reacts to Young Thug Being ...,Cop Your XXL Merch NowYoung Thug is back in th...,"young thug, ysl, ysl rico, rico, brian steel"
3,2024-11-11,Did Young Thug Really Make Those Comments Abou...,Young Thug Tweet Puzzles Fans The current stat...,Cop Your XXL Merch NowYoung Thug's recent twee...,"young thug, gunna, ysl, ysl rico, rico"
4,2024-11-13,We’re So Confused About What’s Going on With Y...,The curious case of the status between Young T...,Cop Your XXL Merch NowThe curious case of the ...,"young thug, gunna, ysl, ysl rico, rico, williams"
5,2024-11-13,Joe Budden Thinks Young Thug Would Be a Fool t...,Read More: We're So Confused About What's Goin...,Cop Your XXL Merch NowJoe Budden insists it wo...,"young thug, gunna, ysl, ysl rico, rico"
...,...,...,...,...,...
1316,2022-02-18,The Break Presents – Saucy Santana,"Such is the case with Saucy Santana, who was o...","Some people are just made for stardom, and exc...","gunna, kendrick"
1317,2022-02-21,Here’s One Great Song From Great Hip-Hop Album...,"On the other side of the coin, the somber yet ...",Even after Nas stated hip-hop is dead years ag...,"young thug, gunna"
1321,2020-11-20,Young Thug Confirms Paying Lil Baby to Focus o...,"While chatting with Young Thug, Tip asked the ...",Lil Baby humbly attributed his start in the ra...,young thug
1323,2020-11-27,Young Thug Says He’s Never Paid Attention to A...,One would think the eclectic styles of Young T...,One would think the eclectic styles of Young T...,young thug


In [965]:
xxl_sentiment.to_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/Cleaned Data/Sentiment Analysis/Magazine/xxl.csv')
xxl_topicmodel.to_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/Cleaned Data/Topic Modeling/Magazine/xxl.csv')

# 4 Internet

## 4.1 Reddit

In [966]:
connection = sqlite3.connect('/Users/sagnikchakravarty/Documents/Young Thug/Data/UnCleaned Data/Internet/subredditdata.db')
cursor = connection.cursor()
tables_query = "SELECT name FROM sqlite_master WHERE type='table';"
cursor.execute(tables_query)
tables = cursor.fetchall()
print("Tables in the database:")
for table in tables:
    print(table[0])

youngthug_reddit = pd.read_sql_query('SELECT * FROM YoungThug;', connection)
gunna_reddit = pd.read_sql_query('SELECT * FROM Gunna', connection)
rap_reddit = pd.read_sql_query('SELECT * FROM rap', connection)
atlantology_reddit = pd.read_sql_query('SELECT * FROM atlantology', connection)

connection.close()

Tables in the database:
YoungThug
Gunna
rap
Atlantology


In [967]:
def reddit_clean(df, i):
    n = df.shape
    df['Published Date'] = pd.to_datetime(df['date_utc']) + pd.to_timedelta(df['timestamp'], unit='s')
    df.drop(['date_utc', 'timestamp', 'url'], axis=1, inplace=True)
    df.rename({'keyword':'search term'}, axis=1, inplace=True)
    df['text'] = df['text'].astype('str')
    df['title'] = df['title'].astype('str')
    df['complete'] = df['text'] +' '+ df['title']
    df['Keywords'] = df['complete'].apply(lambda x: find_matching_keywords(x, keywords))
    df.drop('complete', axis=1, inplace=True)
    df.dropna(subset=['Keywords'], inplace=True)
    df = df[['Published Date', 'title', 'text', 'comments', 'subreddit', 'search term', 'Keywords']]
    print(f'Data reduced for {str(tables[i])}:\t ', n[0] - df.shape[0])
    return df

In [968]:
youngthug_reddit = reddit_clean(youngthug_reddit, 0)
gunna_reddit = reddit_clean(gunna_reddit, 1)
rap_reddit = reddit_clean(rap_reddit, 2)
atlantology_reddit = reddit_clean(atlantology_reddit, 3)

Data reduced for ('YoungThug',):	  203
Data reduced for ('Gunna',):	  184
Data reduced for ('rap',):	  88
Data reduced for ('Atlantology',):	  215


In [969]:
reddit = pd.concat([youngthug_reddit, gunna_reddit, rap_reddit, atlantology_reddit])

In [970]:
reddit_sentiment = sentiment_cleaner.clean_dataframe(reddit.copy(), ['title', 'text'])
reddit_topicmodel = topicmodel_cleaner.clean_dataframe(reddit.copy(), ['title', 'text'])

cleaning data title:   0%|          | 0/2076 [00:00<?, ?it/s]

cleaning data text:   0%|          | 0/2076 [00:00<?, ?it/s]

Successfully cleaned data frame.


cleaning data title:   0%|          | 0/2076 [00:00<?, ?it/s]

cleaning data text:   0%|          | 0/2076 [00:00<?, ?it/s]

Successfully cleaned data frame.


In [971]:
reddit_sentiment.to_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/Cleaned Data/Sentiment Analysis/Internet/reddit.csv')
reddit_topicmodel.to_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/Cleaned Data/Topic Modeling/Internet/reddit.csv')

## 4.2 YouTube
### 4.2.1 Law&Order

In [972]:
def YouTube_Cleaner(df):
    df.rename({'likes_x':'video likes', 'likes_y': 'comment likes', 'comments': 'total comment'}, axis=1, inplace=True)
    df['title'] = df['title'].astype('str')
    df['comment'] = df['comment'].astype('str')
    df['published_at'] = df['published_at'].apply(parse_date)
    df['upload_date'] = df['upload_date'].apply(parse_date)
    df['views'] = df['views'].fillna(0).astype('float64')
    df['total comment'] = df['total comment'].fillna(0).astype('float64')
    df['comment likes'] = df['comment likes'].fillna(0).astype('float64')
    df['replies'] = df['replies'].fillna(0).astype('float64')
    df['complete'] = df['title'] +' '+ df['comment']
    df['Keywords'] = df['complete'].apply(lambda x: find_matching_keywords(x, keywords))
    df.drop('complete', axis=1, inplace=True)
    return df
    

In [973]:
LawAndOrder = pd.read_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/UnCleaned Data/Internet/YouTube/youtube_young_thug_full_data.csv', low_memory=False)
LawAndOrder = YouTube_Cleaner(LawAndOrder)
LawAndOrder

,video_id,title,views,video likes,total comment,upload_date,comment,comment likes,replies,published_at,Keywords
0,5IwZ266l564,"Young Thug, YSL trial stream Day 1",95051.0,847,187.0,2023-11-27 23:04:04+00:00,Her face is the definition of hate 😤😤😤,0.0,0.0,2024-12-13 03:20:50+00:00,"young thug, ysl"
1,5IwZ266l564,"Young Thug, YSL trial stream Day 1",95051.0,847,187.0,2023-11-27 23:04:04+00:00,First time watching from the beginning,2.0,0.0,2024-12-07 02:14:54+00:00,"young thug, ysl"
2,5IwZ266l564,"Young Thug, YSL trial stream Day 1",95051.0,847,187.0,2023-11-27 23:04:04+00:00,Love was always a lying B!!!!!!!!!!!!!!!!!! I ...,1.0,0.0,2024-11-27 05:13:45+00:00,"young thug, ysl"
3,5IwZ266l564,"Young Thug, YSL trial stream Day 1",95051.0,847,187.0,2023-11-27 23:04:04+00:00,My first time watching opening DA LOVE was sne...,2.0,0.0,2024-11-10 23:41:25+00:00,"young thug, ysl"
4,5IwZ266l564,"Young Thug, YSL trial stream Day 1",95051.0,847,187.0,2023-11-27 23:04:04+00:00,The strength of the wolf.. sit down love. Sh...,1.0,0.0,2024-11-09 23:39:06+00:00,"young thug, ysl"
...,...,...,...,...,...,...,...,...,...,...,...
24697,zrbKIgY6Qcc,"Top Intense, Heated Moments in Young Thug’s Ri...",175254.0,2530,1122.0,2024-06-20 21:00:38+00:00,Why does the judge not let anyone speak then l...,203.0,10.0,2024-06-20 21:06:50+00:00,"young thug, rico"
24698,zrbKIgY6Qcc,"Top Intense, Heated Moments in Young Thug’s Ri...",175254.0,2530,1122.0,2024-06-20 21:00:38+00:00,No ya ain&#39;t!,3.0,0.0,2024-06-20 21:06:30+00:00,"young thug, rico"
24699,zrbKIgY6Qcc,"Top Intense, Heated Moments in Young Thug’s Ri...",175254.0,2530,1122.0,2024-06-20 21:00:38+00:00,This Judge is a disgrace,338.0,6.0,2024-06-20 21:05:03+00:00,"young thug, rico"
24700,zrbKIgY6Qcc,"Top Intense, Heated Moments in Young Thug’s Ri...",175254.0,2530,1122.0,2024-06-20 21:00:38+00:00,1st here,3.0,0.0,2024-06-20 21:02:17+00:00,"young thug, rico"


In [974]:
LawAndOrder_sentiment = sentiment_cleaner.clean_dataframe(LawAndOrder.copy(), ['title', 'comment'])
LawAndOrder_topicmodels = topicmodel_cleaner.clean_dataframe(LawAndOrder.copy(), ['title', 'comment'])

cleaning data title:   0%|          | 0/24702 [00:00<?, ?it/s]

cleaning data comment:   0%|          | 0/24702 [00:00<?, ?it/s]

Successfully cleaned data frame.


cleaning data title:   0%|          | 0/24702 [00:00<?, ?it/s]

cleaning data comment:   0%|          | 0/24702 [00:00<?, ?it/s]

Successfully cleaned data frame.


In [975]:
LawAndOrder_sentiment.to_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/Cleaned Data/Sentiment Analysis/Internet/LawAndOrder_YouTube.csv')
LawAndOrder_topicmodels.to_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/Cleaned Data/Topic Modeling/Internet/LawAndOrder_YouTube.csv')

### 4.2.2 Artist Videos

#### 4.2.2.1 Young Thug

In [1029]:
def YouTube_Artist_Cleaner(df):
    df = df.rename({'published_at': 'upload_date',
               'timestamp': 'published at',
               'text': 'comment',
               'video_title': 'title',
               'view_count': 'views',
               'like_count': 'video likes',
               'comment_count': 'total comment',
               'likes': 'comment likes',
               }, axis=1)
    df['title'] = df['title'].astype('str')
    df['comment'] = df['comment'].astype('str')
    df['description'] = df['description'].astype('str')
    df['video_duration'] = df['video_duration'].apply(lambda x: isodate.parse_duration(x))
    df['published at'] = df['published at'].apply(parse_date)
    df['upload_date'] = df['upload_date'].apply(parse_date)
    df['views'] = df['views'].fillna(0).astype('float64')
    df['total comment'] = df['total comment'].fillna(0).astype('float64')
    df['comment likes'] = df['comment likes'].fillna(0).astype('float64')
    df['Keywords'] = df['comment'].apply(lambda x: find_matching_keywords(x, keywords))
    df = df[['video_id', 
             'channel_title', 
             'video_duration', 
             'upload_date',
             'title',
             'description',
             'views',
             'video likes',
             'total comment',
             'published at',
             'comment',
             'comment likes',
             'Keywords']].copy()
    return df

In [1031]:
youngthug_youtube_stats = pd.read_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/UnCleaned Data/Internet/YouTube/Artist Channels/youngthug_youtube_video_stats.csv', engine = 'python', index_col = 0)
youngthug_youtube_comments = pd.read_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/UnCleaned Data/Internet/YouTube/Artist Channels/youngthug_youtube_video_comments.csv', engine = 'python', index_col = 0)
youngthug_youtube = pd.merge(youngthug_youtube_stats, youngthug_youtube_comments, on='video_id', how='left')

#### 4.2.2.2 Gunna

In [1032]:
gunna_youtube_stats = pd.read_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/UnCleaned Data/Internet/YouTube/Artist Channels/gunna_youtube_video_stats.csv', engine = 'python', index_col = 0)
gunna_youtube_comments = pd.read_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/UnCleaned Data/Internet/YouTube/Artist Channels/gunna_youtube_video_comments.csv', engine = 'python', index_col = 0)
gunna_youtube = pd.merge(gunna_youtube_stats, gunna_youtube_comments, on='video_id', how='left')

#### 4.2.2.3 YFN Lucci

In [1034]:
yfn_youtube_stats = pd.read_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/UnCleaned Data/Internet/YouTube/Artist Channels/yfnlucci_youtube_video_stats.csv', engine = 'python', index_col = 0)
yfn_youtube_comment = pd.read_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/UnCleaned Data/Internet/YouTube/Artist Channels/yfnlucci_youtube_video_comments.csv', engine = 'python', index_col = 0)
yfn_youtube = pd.merge(yfn_youtube_stats, yfn_youtube_comment, on='video_id', how='left')

In [1035]:
YouTube_Artist = pd.concat([youngthug_youtube, gunna_youtube, yfn_youtube])
YouTube_Artist = YouTube_Artist_Cleaner(YouTube_Artist)

In [1036]:
YouTube_Artist_Sentiment = sentiment_cleaner.clean_dataframe(YouTube_Artist.copy(), 'comment')
YouTube_Artist_TopicModels = topicmodel_cleaner.clean_dataframe(YouTube_Artist_Sentiment.copy(), 'comment')

cleaning data comment:   0%|          | 0/190645 [00:00<?, ?it/s]

Successfully cleaned data frame.


cleaning data comment:   0%|          | 0/190645 [00:00<?, ?it/s]

Successfully cleaned data frame.


In [1038]:
YouTube_Artist_Sentiment.to_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/Cleaned Data/Sentiment Analysis/Internet/YouTube Artist.csv')
YouTube_Artist_Sentiment.to_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/Cleaned Data/Topic Modeling/Internet/YouTube Artist.csv')

## 4.3 Instagram

In [1044]:
instagram = pd.read_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/UnCleaned Data/Internet/ysl_trial_top_instagram.csv', index_col = 0)
instagram

,caption,like_count,permalink,timestamp,comments_count,id
0,#youngthug’s lawyer now representing #problemc...,1820.0,https://www.instagram.com/p/DDxL4hlR6lR/,2024-12-19T17:35:30+0000,65,18013719890662275
1,#YoungThug has filed a new motion to be given ...,NaN,https://www.instagram.com/p/C9yP2ThpgSt/,2024-07-24T00:21:01+0000,18,18008405282548887
2,In YSL's RICO Case The Judge Reads Lyrics Off ...,621.0,https://www.instagram.com/p/CnHbvp3jENH/,2023-01-07T13:47:30+0000,80,18266528860114452
3,YSL Co-Founder Demonstrates Gang Signs in Cour...,36.0,https://www.instagram.com/p/C1pdT29PUHi/,2024-01-03T18:14:20+0000,3,17908442528808076
4,Looks like Mr. Thugger is already motivating t...,NaN,https://www.instagram.com/p/DCFWvBzzQkx/,2024-11-07T20:32:21+0000,8,18052871134937422
5,We all know that #Wham is the nickname that #Y...,NaN,https://www.instagram.com/p/C8urrFWPDOU/,2024-06-27T18:35:48+0000,1,18016393640380619
6,🚨 BREAKING NEWS 🚨 Judge Ural Glanville presidi...,NaN,https://www.instagram.com/p/C9dIZoiB8iW/,2024-07-15T19:31:54+0000,0,18023226593256628
7,Mannnn. What do yall think about this? 🤔\n#you...,1879.0,https://www.instagram.com/p/C8uhezNCXBQ/,2024-06-27T17:06:45+0000,123,18039000883787349
8,#ysltrial isn’t over yet and continues Monday ...,604.0,https://www.instagram.com/p/DB4dWq7RSG1/,2024-11-02T20:20:04+0000,12,18317631610095385
9,This the new judge on #YoungThug’s Rico court ...,NaN,https://www.instagram.com/p/C9i0Q6hJlK-/,2024-07-18T00:31:23+0000,4,18038591872958710


In [1045]:
instagram['like_count'] = instagram['like_count'].fillna(0).astype('float64')
instagram['timestamp'] = instagram['timestamp'].apply(parse_date)
instagram.drop('permalink', axis=1, inplace=True)
instagram['Keywords'] = instagram['caption'].apply(lambda x: find_matching_keywords(x, keywords))
instagram['comments_count'] = instagram['comments_count'].fillna(0).astype('float64')
instagram['caption'] = instagram['caption'].astype('str')
instagram = instagram[['timestamp', 'caption', 'like_count', 'comments_count']].copy()
instagram

,timestamp,caption,like_count,comments_count
0,2024-12-19 17:35:30+00:00,#youngthug’s lawyer now representing #problemc...,1820.0,65.0
1,2024-07-24 00:21:01+00:00,#YoungThug has filed a new motion to be given ...,0.0,18.0
2,2023-01-07 13:47:30+00:00,In YSL's RICO Case The Judge Reads Lyrics Off ...,621.0,80.0
3,2024-01-03 18:14:20+00:00,YSL Co-Founder Demonstrates Gang Signs in Cour...,36.0,3.0
4,2024-11-07 20:32:21+00:00,Looks like Mr. Thugger is already motivating t...,0.0,8.0
5,2024-06-27 18:35:48+00:00,We all know that #Wham is the nickname that #Y...,0.0,1.0
6,2024-07-15 19:31:54+00:00,🚨 BREAKING NEWS 🚨 Judge Ural Glanville presidi...,0.0,0.0
7,2024-06-27 17:06:45+00:00,Mannnn. What do yall think about this? 🤔\n#you...,1879.0,123.0
8,2024-11-02 20:20:04+00:00,#ysltrial isn’t over yet and continues Monday ...,604.0,12.0
9,2024-07-18 00:31:23+00:00,This the new judge on #YoungThug’s Rico court ...,0.0,4.0


In [1046]:
instagram_sentiment = sentiment_cleaner.clean_dataframe(instagram, 'caption')
instagram_topicmodels = topicmodel_cleaner.clean_dataframe(instagram_sentiment, 'caption')

cleaning data caption:   0%|          | 0/25 [00:00<?, ?it/s]

Successfully cleaned data frame.


cleaning data caption:   0%|          | 0/25 [00:00<?, ?it/s]

Successfully cleaned data frame.


In [1047]:
instagram_sentiment.to_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/Cleaned Data/Sentiment Analysis/Internet/instagram.csv')
instagram_topicmodels.to_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/Cleaned Data/Topic Modeling/Internet/instagram.csv')

## 4.4 Spotify
### 4.4.1 Artist

In [1056]:
youngthug_spotify = pd.read_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/UnCleaned Data/Internet/Spotify/YoungThug_Album_SongDetails.csv', index_col = 0)
gunna_spotify = pd.read_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/UnCleaned Data/Internet/Spotify/Gunna_Album_SongDetails.csv', index_col = 0)
yfn_spotify = pd.read_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/UnCleaned Data/Internet/Spotify/YfnLucci_Album_SongDetails.csv', index_col = 0)
youngthug_spotify['Artist'] = 'Young Thug'
gunna_spotify['Artist'] = 'Gunna'
yfn_spotify['Artist'] = 'YFN Lucci'
spotify_artist = pd.concat([youngthug_spotify, gunna_spotify, yfn_spotify])
spotify_artist.to_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/Cleaned Data/Sentiment Analysis/Internet/spotify_artist.csv')
spotify_artist

,Name,Available Market,explicit,duration,album_name,Song ID,Popularity,Release Date,Album ID,Track Number,Artist
0,Jonesboro,184,True,163800,BUSINESS IS BUSINESS (Metro's Version),3sSlrxRD9X1jYCzBtbbiwg,42,2023-06-26,0z2a9VgdVmkr0DInVJUgu6,1,Young Thug
1,Mad Dog,184,True,229000,BUSINESS IS BUSINESS (Metro's Version),332TO0w5iJ8h4h55Qq5a0R,38,2023-06-26,0z2a9VgdVmkr0DInVJUgu6,2,Young Thug
2,Uncle M,184,True,140000,BUSINESS IS BUSINESS (Metro's Version),2BR4cE9udCDcvZ2HtuLv3N,38,2023-06-26,0z2a9VgdVmkr0DInVJUgu6,3,Young Thug
3,Want Me Dead (feat. 21 Savage),184,True,194706,BUSINESS IS BUSINESS (Metro's Version),4r4rpFbzk8LDKosME0ZW1C,43,2023-06-26,0z2a9VgdVmkr0DInVJUgu6,4,Young Thug
4,Cars Bring Me Out (feat. Future),184,True,202933,BUSINESS IS BUSINESS (Metro's Version),0GY3LiAOOOYIIDv53TXbjR,39,2023-06-26,0z2a9VgdVmkr0DInVJUgu6,5,Young Thug
...,...,...,...,...,...,...,...,...,...,...,...
192,YEA YEA,185,True,215171,YEA YEA,2vkmfq2Mi3lxp0LFrfK97I,11,2021-02-04,0bkIQcaGoDA0fS2pA3pEBY,1,YFN Lucci
193,I Gotcha,185,True,161375,I Gotcha,3r56zMIKGtWWAr9F1hQz98,26,2021-02-02,535bn6GbeouGSRDBjqkhsb,1,YFN Lucci
194,Safe,185,True,210040,Safe,610j23V23O9DMcds7Eml35,7,2021-01-22,7nWJq9yNfP1gmcFulyCbkU,1,YFN Lucci
195,Hustle (feat. YFN Lucci & Yungeen Ace),185,True,201329,Hustle (feat. YFN Lucci & Yungeen Ace),1kEXulOQtLjPvwQlnoksBr,21,2020-12-18,6n0xa1uVVUG7egMzvG3GsR,1,YFN Lucci


### 4.4.2 Podcast

In [1067]:
podcast = pd.read_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/UnCleaned Data/Internet/Spotify/Podcast/KingSlimeVideo.csv', index_col = 0)
podcast['Keywords'] = podcast['description'].apply(lambda x: find_matching_keywords(x, keywords))
podcast_sentiment = sentiment_cleaner.clean_dataframe(podcast, 'description')
podcast_topicmodels = topicmodel_cleaner.clean_dataframe(podcast_sentiment, 'description')

cleaning data description:   0%|          | 0/16 [00:00<?, ?it/s]

Successfully cleaned data frame.


cleaning data description:   0%|          | 0/16 [00:00<?, ?it/s]

Successfully cleaned data frame.


In [1068]:
podcast_topicmodels.to_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/Cleaned Data/Topic Modeling/Internet/KingSlime.csv')
podcast_sentiment.to_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/Cleaned Data/Sentiment Analysis/Internet/KingSlime.csv')

## 4.5 Twitter (X)
### 4.5.1 ThuggerDaily

In [1072]:
thuggerdaily = pd.read_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/UnCleaned Data/Internet/Twitter/ThuggerDailyTwitter.csv')
thuggerdaily

,Name,Handle,Timestamp,Verified,Content,Comments,Retweets,Likes,Analytics,Tags,Mentions,Emojis,Profile Image,Tweet Link,Tweet ID
0,THUGGERDAILY ひ,@ThuggerDaily,2024-12-07T13:06:02.000Z,True,,8,4,262,11K,[],[],"['\\U0001f60e', '\\U0001f60e', '\\u2764\\ufe0f']",https://pbs.twimg.com/profile_images/169659762...,https://x.com/ThuggerDaily/status/186538226088...,tweet_id:1865382260884549891
1,THUGGERDAILY ひ,@ThuggerDaily,2024-12-07T00:42:45.000Z,True,the shell kel stuff is overblown there’s no co...,16,5,111,14K,[],[],[],https://pbs.twimg.com/profile_images/169659762...,https://x.com/ThuggerDaily/status/186519520538...,tweet_id:1865195205386789325
2,THUGGERDAILY ひ,@ThuggerDaily,2024-12-06T22:17:14.000Z,True,But Slato's potential argument here is that th...,15,6,202,35K,[],[],[],https://pbs.twimg.com/profile_images/169659762...,https://x.com/ThuggerDaily/status/186515858666...,tweet_id:1865158586663346622
3,THUGGERDAILY ひ,@ThuggerDaily,2024-12-06T22:10:25.000Z,True,"Found the hang up in this... I said it was ""ti...",6,5,175,25K,[],[],[],https://pbs.twimg.com/profile_images/169659762...,https://x.com/ThuggerDaily/status/186515687042...,tweet_id:1865156870429614271
4,THUGGERDAILY ひ,@ThuggerDaily,2024-12-06T21:53:06.000Z,True,during selection he had his own prescription p...,3,0,72,5.5K,[],[],[],https://pbs.twimg.com/profile_images/169659762...,https://x.com/ThuggerDaily/status/186515251255...,tweet_id:1865152512551964793
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3387,THUGGERDAILY ひ,@ThuggerDaily,2023-01-02T19:24:53.000Z,True,NaN,5,18,244,24K,[],[],[],https://pbs.twimg.com/profile_images/169659762...,https://x.com/ThuggerDaily/status/160999415316...,tweet_id:1609994153168982017
3388,THUGGERDAILY ひ,@ThuggerDaily,2023-01-02T18:17:16.000Z,True,Ignore the clout chasers! Take it from thugs s...,6,105,1.1K,103K,[],[],[],https://pbs.twimg.com/profile_images/169659762...,https://x.com/ThuggerDaily/status/160997713945...,tweet_id:1609977139457048577
3389,THUGGERDAILY ひ,@ThuggerDaily,2023-01-01T18:40:40.000Z,True,Every defendant has their own charges based of...,3,19,339,16K,[],[],[],https://pbs.twimg.com/profile_images/169659762...,https://x.com/ThuggerDaily/status/160962063822...,tweet_id:1609620638221717506
3390,THUGGERDAILY ひ,@ThuggerDaily,2023-01-01T18:33:57.000Z,True,"Fact Check: \n\n- Currently, young thug doesn'...",19,95,1.3K,97K,[],[],[],https://pbs.twimg.com/profile_images/169659762...,https://x.com/ThuggerDaily/status/160961894847...,tweet_id:1609618948471009281


In [1091]:
def convert_shorthand_to_int(shorthand):
    """
    Convert shorthand numbers like '11k', '1.2M', '3B' to integers.
    """
    shorthand = str(shorthand)
    shorthand = shorthand.lower()  # Convert to lowercase for consistency
    if 'k' in shorthand:
        return int(float(shorthand.replace('k', '')) * 1000)
    elif 'm' in shorthand:
        return int(float(shorthand.replace('m', '')) * 1000000)
    elif 'b' in shorthand:
        return int(float(shorthand.replace('b', '')) * 1000000000)
    else:
        return int(shorthand)  # If no shorthand, return as integer

def twitter_cleaner(df, bool = False):
    df['Content'] = df['Content'].astype(str)
    df['Timestamp'] = df['Timestamp'].apply(parse_date)
    df['Tweets'] = df.apply(lambda row: f"{row['Content']} {row['Emojis']}" if pd.notna(row['Content']) else row['Emojis'], axis=1)
    df['Comments'] = df['Comments'].apply(convert_shorthand_to_int)
    df['Retweets'] = df['Retweets'].apply(convert_shorthand_to_int)
    df['Likes'] = df['Likes'].apply(convert_shorthand_to_int)
    df['Analytics'] = df['Analytics'].apply(convert_shorthand_to_int)
    df['Keywords'] = df['Tweets'].apply(lambda x: find_matching_keywords(x, keywords))
    if bool:
        df = df[['Timestamp','Name', 'Handle', 'Verified', 'Tweets', 'Comments', 'Retweets', 'Likes', 'Analytics', 'Tags', 'Mentions', 'Keywords']].copy()
        return df
    df = df[['Timestamp', 'Tweets', 'Comments', 'Retweets', 'Likes', 'Analytics', 'Keywords']].copy()
    return df

In [1074]:
thuggerdaily = twitter_cleaner(thuggerdaily)

In [1075]:
thuggerdaily.to_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/Cleaned Data/Sentiment Analysis/Internet/thuggerdaily_twitter.csv')
thuggerdaily.to_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/Cleaned Data/Topic Modeling/Internet/thuggerdaily_twitter.csv')

In [1076]:
thuggerdaily

,Timestamp,Tweets,Comments,Retweets,Likes,Analytics,Keywords
0,2024-12-07 13:06:02+00:00,"['\\U0001f60e', '\\U0001f60e', '\\u2764\\uf...",8,4,262,11000,None
1,2024-12-07 00:42:45+00:00,the shell kel stuff is overblown there’s no co...,16,5,111,14000,ysl
2,2024-12-06 22:17:14+00:00,But Slato's potential argument here is that th...,15,6,202,35000,"shannon, glanville"
3,2024-12-06 22:10:25+00:00,"Found the hang up in this... I said it was ""ti...",6,5,175,25000,"shannon, glanville"
4,2024-12-06 21:53:06+00:00,during selection he had his own prescription p...,3,0,72,5500,None
...,...,...,...,...,...,...,...
3387,2023-01-02 19:24:53+00:00,nan [],5,18,244,24000,None
3388,2023-01-02 18:17:16+00:00,Ignore the clout chasers! Take it from thugs s...,6,105,1100,103000,None
3389,2023-01-01 18:40:40+00:00,Every defendant has their own charges based of...,3,19,339,16000,rico
3390,2023-01-01 18:33:57+00:00,"Fact Check: \n\n- Currently, young thug doesn'...",19,95,1300,97000,"young thug, rico"


### 4.4.2 YSL hashtag

In [1092]:
ysl = pd.read_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/UnCleaned Data/Internet/Twitter/twittwer22-24.csv')

ysl = twitter_cleaner(ysl, bool = True)
ysl

,Timestamp,Name,Handle,Verified,Tweets,Comments,Retweets,Likes,Analytics,Tags,Mentions,Keywords
0,2024-12-27 03:50:39+00:00,KZ,@kz3354,False,"#ysltrial watchers, Shell Kel‚Äôs interview ju...",0,0,0,1,['#ysltrial'],[],ysl
1,2024-12-27 03:34:07+00:00,chrissy paradis,@chrissyparadis,True,"ADA Love, Nathan Wade, Judge Glanville & Fani ...",0,0,1,64,"['#ysltrial', '#ysl']",[],"ysl, glanville"
2,2024-12-27 00:43:45+00:00,Valerie,@queen__valerie_,True,The longest trial is GA history has to be the ...,1,0,5,228,"['#ysltrial', '#netflix']",[],ysl
3,2024-12-27 00:11:17+00:00,Valerie,@queen__valerie_,True,Agree 100%. It was like watching a very broad ...,3,0,3,518,['#ysltrial'],[],ysl
4,2024-12-26 23:45:13+00:00,Sunglasses x Sneakers,@SceneByAshlix,False,Judge Whitaker anytime Attorney Love opened he...,0,0,1,33,['#YSLTrial'],[],"ysl, whitaker"
...,...,...,...,...,...,...,...,...,...,...,...,...
743,2022-12-22 04:48:27+00:00,King Jaffe Joseph,@King_JaffeJoe,False,Latest On YSL Trial | Young Thug's Brother Jo...,0,1,2,1600,"['#torylaneztrial', '#youngthugtrial', '#ysltr...",['@YouTube'],"young thug, gunna, ysl"
744,2022-12-21 21:55:44+00:00,tee dee,@Agnb_223Twon,False,Judge Phillip Banks ain‚Äôt taking it easy on ...,0,0,1,134,"['#youngthug', '#ysl', '#ysltrial']",[],ysl
745,2022-12-19 17:01:40+00:00,Ray Ray Wick,@KingBigWuan,False,at home smoking a fat blunt while his ‚Äúbro‚...,0,0,0,78,['#ysltrial'],['@1GunnaGunna'],ysl
746,2022-12-22 15:55:51+00:00,YSL TRIAL TRACKER,@YSLTRIALTRACKER,False,"Only 24 entries thus far, don‚Äôt miss your ch...",0,2,4,2000,"['#ysltrial', '#freeyoungthug']",['@9PM'],ysl


In [1093]:
ysl_sentiment = sentiment_cleaner.clean_dataframe(ysl, 'Tweets')
ysl_topicmodel = topicmodel_cleaner.clean_dataframe(ysl_sentiment, 'Tweets')

cleaning data Tweets:   0%|          | 0/748 [00:00<?, ?it/s]

Successfully cleaned data frame.


cleaning data Tweets:   0%|          | 0/748 [00:00<?, ?it/s]

Successfully cleaned data frame.


In [1095]:
ysl_sentiment.to_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/Cleaned Data/Sentiment Analysis/Internet/ysl.csv')
ysl_topicmodel.to_csv('/Users/sagnikchakravarty/Documents/Young Thug/Data/Cleaned Data/Topic Modeling/Internet/ysl.csv')